02/10/2025 verzija

In [ ]:
# !pip uninstall pybamm -y

In [ ]:
# ! pip install -e. # go to file install_pybamm.ipynb

In [ ]:
# aktiviraj myenv okolje
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize

In [ ]:
# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

## Nova koda

In [ ]:
def my_current(t):
    return pybamm.sin( 2*np.pi * t *10)/1e1 # amplitude 0.1 A, frequency 10 Hz

t_eval = np.linspace(0, 1, 1000)

In [ ]:
# model, ki ima le mehansko degradacijo
# model_DFN = pybamm.lithium_ion.BasicDFN()

model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        name="Meh.",)

param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0},
    check_already_exists=False  
)
param0["Current function [A]"] = my_current

param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e9},
    check_already_exists=False  
)
param1["Current function [A]"] = my_current

param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 2e9},
    check_already_exists=False  
)
param2["Current function [A]"] = my_current

var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

# experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  

sim_DFN_0 = pybamm.Simulation(model= model_DFN, parameter_values=param0, var_pts=var_pts, solver=solver, 
                              #experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model_DFN, parameter_values=param1, var_pts=var_pts, solver=solver, 
                              #experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model_DFN, parameter_values=param2, var_pts=var_pts, solver=solver,
                              #experiment=experiment 
                              )

######################
# t_eval = [0, 3600]  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
# print("0 Pa: ", param0["Open-circuit voltage at 0% SOC [V]"])
# print("1 GPa: ", param1["Open-circuit voltage at 0% SOC [V]"])
# print("2 GPa: ", param2["Open-circuit voltage at 0% SOC [V]"])

In [ ]:
# print("0 Pa: ", param0["Open-circuit voltage at 100% SOC [V]"])
# print("1 GPa: ", param1["Open-circuit voltage at 100% SOC [V]"])
# print("2 GPa: ", param2["Open-circuit voltage at 100% SOC [V]"])

In [ ]:
voltage = sol_DFN["Voltage [V]"].entries 
voltage_1 = sol_DFN_1["Voltage [V]"].entries
voltage_2 = sol_DFN_2["Voltage [V]"].entries
x = sol_DFN["Time [s]"].entries
x_1 = sol_DFN_1["Time [s]"].entries
x_2 = sol_DFN_2["Time [s]"].entries

plt.figure(figsize=fig_size)
plt.plot(x, voltage, label="0 Pa")
plt.plot(x_1, voltage_1, label="1e9 Pa", alpha=0.8, linestyle='--')
plt.plot(x_2, voltage_2, label="2e9 Pa", alpha=0.8, linestyle='--')
plt.xlabel("Time [s]")
plt.ylabel("Voltage [V]")
plt.legend(loc ='upper right', bbox_to_anchor=(1.35, 1))

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
# Q_max = Q_discharged[-1] 
Q_max = 2.28*3600
DOD = (Q_discharged / Q_max)

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
# Q_max = Q_discharged[-1] 
Q_max = 2.28*3600
DOD_1 = (Q_discharged / Q_max)

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
# Q_max = Q_discharged[-1] 
Q_max = 2.28*3600
DOD_2 = (Q_discharged / Q_max)

x_val = [DOD, 
         DOD_1,
        DOD_2
         ]
y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e9 Pa", 
                                        "$\sigma_{\mathrm{h}}=$2e9 Pa"
                                        
                                        ][i], 
                                        linestyle=["-", "--", ":", "-", "-", "-", "-", "-", "-", "-"][i])
    
# plt.xlim(3000, 3600)
# plt.ylim(3.5, 3.6)
plt.xlabel("DOD [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

if save_fig:
    plt.savefig("graphs/OCV_i0.eps", format='eps', dpi=600, bbox_inches='tight')


## Stara koda

In [ ]:
# model, ki ima le mehansko degradacijo
model_DFN = pybamm.lithium_ion.BasicDFN()

param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0},
    check_already_exists=False  
)


param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e9},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 2e9},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model_DFN, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model_DFN, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model_DFN, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)

x_val = [DOD, 
         DOD_1,
        DOD_2
         ]
y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e9 Pa", 
                                        "$\sigma_{\mathrm{h}}=$2e9 Pa"
                                        ][i], 
                                        linestyle=["-", "--", ":", "-", "-", "-", "-", "-", "-", "-"][i])
    
# plt.xlim(3000, 3600)
# plt.ylim(3.5, 3.6)
plt.xlabel("DOD [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

if save_fig:
    plt.savefig("graphs/OCV_i0.eps", format='eps', dpi=600, bbox_inches='tight')


## EIS

In [ ]:
# import numpy as np
# from scipy import interpolate
# from scipy import optimize
# import pybamm
# import matplotlib.pyplot as plt

$\sigma_h$ = 0e9 Pa

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0 = np.array(Z_0)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0[:, 1], -Z_0[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0[:, 1] + 1j * Z_0[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0[:, 1] + 1j * Z_0[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0Pa_highPrecision.txt", Z_0, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

$\sigma_h$ = 1e9 Pa

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1 = np.array(Z_1)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1[:, 1], -Z_1[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1[:, 1] + 1j * Z_1[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1[:, 1] + 1j * Z_1[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_highPrecision.txt", Z_1, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

$\sigma_h$ = 2e9 Pa

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                     )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_2 = sim_DFN_2.solve(t_eval=t)
   
    simulated_U_L_2 =  sol_DFN_2["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_2) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_2)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_2["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_2
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_2)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2 = np.array(Z_2)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2[:, 1], -Z_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2[:, 1] + 1j * Z_2[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2[:, 1] + 1j * Z_2[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_highPrecision.txt", Z_2, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

In [ ]:
plt.plot(Z_0[:, 1], -Z_0[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa")
plt.plot(Z_1[:, 1], -Z_1[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa")
plt.plot(Z_2[:, 1], -Z_2[:, 2], marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "upper right", bbox_to_anchor=(1.5, 0.9))

# Različni SOC

### $\sigma_h$ = 0e9 Pa

#### SOC = 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc100 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc100.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc100 = np.array(Z_0_soc100)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc100[:, 1], -Z_0_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc100[:, 1] + 1j * Z_0_soc100[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc100[:, 1] + 1j * Z_0_soc100[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc100[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0Pa_soc100_highPrecision.txt", Z_0_soc100, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.9

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc90 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.9)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc90.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc90 = np.array(Z_0_soc90)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc90[:, 1], -Z_0_soc90[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc90[:, 1] + 1j * Z_0_soc90[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc90[:, 1] + 1j * Z_0_soc90[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc90[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc90_highPrecision.txt", Z_0_soc90, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.8

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc80 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.8)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc80.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc80 = np.array(Z_0_soc80)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc80[:, 1], -Z_0_soc80[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc80[:, 1] + 1j * Z_0_soc80[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc80[:, 1] + 1j * Z_0_soc80[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc80[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc80_highPrecision.txt", Z_0_soc80, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.7

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc70 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.7)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc70.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc70 = np.array(Z_0_soc70)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc70[:, 1], -Z_0_soc70[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc70[:, 1] + 1j * Z_0_soc70[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc70[:, 1] + 1j * Z_0_soc70[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc70[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc70_highPrecision.txt", Z_0_soc70, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.6

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc60 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.6)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc60.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc60 = np.array(Z_0_soc60)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc60[:, 1], -Z_0_soc60[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc60[:, 1] + 1j * Z_0_soc60[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc60[:, 1] + 1j * Z_0_soc60[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc60[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc60_highPrecision.txt", Z_0_soc60, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.5

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc50 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=0.5)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc50.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc50 = np.array(Z_0_soc50)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc50[:, 1], -Z_0_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc50[:, 1] + 1j * Z_0_soc50[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc50[:, 1] + 1j * Z_0_soc50[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc50[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0Pa_soc50_highPrecision.txt", Z_0_soc50, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.4

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc40 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.4)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc40.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc40 = np.array(Z_0_soc40)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc40[:, 1], -Z_0_soc40[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc40[:, 1] + 1j * Z_0_soc40[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc40[:, 1] + 1j * Z_0_soc40[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc40[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc40_highPrecision.txt", Z_0_soc40, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.3

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc30 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.3)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc30.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc30 = np.array(Z_0_soc30)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc30[:, 1], -Z_0_soc30[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc30[:, 1] + 1j * Z_0_soc30[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc30[:, 1] + 1j * Z_0_soc30[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc30[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc30_highPrecision.txt", Z_0_soc30, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.2

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc20 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=0.2)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc20.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc20 = np.array(Z_0_soc20)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc20[:, 1], -Z_0_soc20[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc20[:, 1] + 1j * Z_0_soc20[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc20[:, 1] + 1j * Z_0_soc20[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc20[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0Pa_soc20_highPrecision.txt", Z_0_soc20, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0_soc10 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.1)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0_soc10.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0_soc10 = np.array(Z_0_soc10)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0_soc10[:, 1], -Z_0_soc10[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0_soc10[:, 1] + 1j * Z_0_soc10[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0_soc10[:, 1] + 1j * Z_0_soc10[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0_soc10[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_0GPa_soc10_highPrecision.txt", Z_0_soc10, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e9 Pa

#### SOC = 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc100 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=1)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc100.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc100 = np.array(Z_1_soc100)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc100[:, 1], -Z_1_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc100[:, 1] + 1j * Z_1_soc100[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc100[:, 1] + 1j * Z_1_soc100[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc100[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc100_highPrecision.txt", Z_1_soc100, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.9

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc90 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.9)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc90.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc90 = np.array(Z_1_soc90)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc90[:, 1], -Z_1_soc90[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc90[:, 1] + 1j * Z_1_soc90[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc90[:, 1] + 1j * Z_1_soc90[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc90[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc90_highPrecision.txt", Z_1_soc90, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.8

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc80 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.8)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc80.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc80 = np.array(Z_1_soc80)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc80[:, 1], -Z_1_soc80[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc80[:, 1] + 1j * Z_1_soc80[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc80[:, 1] + 1j * Z_1_soc80[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc80[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc80_highPrecision.txt", Z_1_soc80, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.7

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc70 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.7)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc70.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc70 = np.array(Z_1_soc70)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc70[:, 1], -Z_1_soc70[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc70[:, 1] + 1j * Z_1_soc70[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc70[:, 1] + 1j * Z_1_soc70[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc70[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc70_highPrecision.txt", Z_1_soc70, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.6

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc60 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.6)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc60.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc60 = np.array(Z_1_soc60)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc60[:, 1], -Z_1_soc60[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc60[:, 1] + 1j * Z_1_soc60[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc60[:, 1] + 1j * Z_1_soc60[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc60[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc60_highPrecision.txt", Z_1_soc60, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.5

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc50 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.5)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc50.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc50 = np.array(Z_1_soc50)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc50[:, 1], -Z_1_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc50[:, 1] + 1j * Z_1_soc50[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc50[:, 1] + 1j * Z_1_soc50[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc50[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc50_highPrecision.txt", Z_1_soc50, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.4

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc40 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.4)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc40.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc40 = np.array(Z_1_soc40)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc40[:, 1], -Z_1_soc40[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc40[:, 1] + 1j * Z_1_soc40[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc40[:, 1] + 1j * Z_1_soc40[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc40[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc40_highPrecision.txt", Z_1_soc40, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.3

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc30 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.3)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc30.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc30 = np.array(Z_1_soc30)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc30[:, 1], -Z_1_soc30[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc30[:, 1] + 1j * Z_1_soc30[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc30[:, 1] + 1j * Z_1_soc30[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc30[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc30_highPrecision.txt", Z_1_soc30, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.2

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc20 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.2)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc20.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc20 = np.array(Z_1_soc20)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc20[:, 1], -Z_1_soc20[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc20[:, 1] + 1j * Z_1_soc20[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc20[:, 1] + 1j * Z_1_soc20[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc20[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc20_highPrecision.txt", Z_1_soc20, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1_soc10 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 1e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.1)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1_soc10.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1_soc10 = np.array(Z_1_soc10)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1_soc10[:, 1], -Z_1_soc10[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1_soc10[:, 1] + 1j * Z_1_soc10[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1_soc10[:, 1] + 1j * Z_1_soc10[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1_soc10[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_1GPa_soc10_highPrecision.txt", Z_1_soc10, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 2e9 Pa

#### SOC = 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc100 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=1)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc100.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc100 = np.array(Z_2_soc100)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc100[:, 1], -Z_2_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc100[:, 1] + 1j * Z_2_soc100[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc100[:, 1] + 1j * Z_2_soc100[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc100[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc100_highPrecision.txt", Z_2_soc100, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.9

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc90 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.9)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc90.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc90 = np.array(Z_2_soc90)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc90[:, 1], -Z_2_soc90[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc90[:, 1] + 1j * Z_2_soc90[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc90[:, 1] + 1j * Z_2_soc90[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc90[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc90_highPrecision.txt", Z_2_soc90, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.8

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc80 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.8)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc80.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc80 = np.array(Z_2_soc80)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc80[:, 1], -Z_2_soc80[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc80[:, 1] + 1j * Z_2_soc80[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc80[:, 1] + 1j * Z_2_soc80[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc80[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc80_highPrecision.txt", Z_2_soc80, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.7

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc70 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.7)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc70.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc70 = np.array(Z_2_soc70)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc70[:, 1], -Z_2_soc70[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc70[:, 1] + 1j * Z_2_soc70[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc70[:, 1] + 1j * Z_2_soc70[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc70[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc70_highPrecision.txt", Z_2_soc70, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.6

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc60 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.6)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc60.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc60 = np.array(Z_2_soc60)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc60[:, 1], -Z_2_soc60[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc60[:, 1] + 1j * Z_2_soc60[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc60[:, 1] + 1j * Z_2_soc60[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc60[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc60_highPrecision.txt", Z_2_soc60, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.5

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc50 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.5)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc50.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc50 = np.array(Z_2_soc50)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc50[:, 1], -Z_2_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc50[:, 1] + 1j * Z_2_soc50[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc50[:, 1] + 1j * Z_2_soc50[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc50[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc50_highPrecision.txt", Z_2_soc50, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.4

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc40 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.4)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc40.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc40 = np.array(Z_2_soc40)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc40[:, 1], -Z_2_soc40[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc40[:, 1] + 1j * Z_2_soc40[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc40[:, 1] + 1j * Z_2_soc40[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc40[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc40_highPrecision.txt", Z_2_soc40, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.3

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc30 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.3)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc30.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc30 = np.array(Z_2_soc30)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc30[:, 1], -Z_2_soc30[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc30[:, 1] + 1j * Z_2_soc30[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc30[:, 1] + 1j * Z_2_soc30[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc30[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc30_highPrecision.txt", Z_2_soc30, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.2

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc20 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.2)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc20.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc20 = np.array(Z_2_soc20)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc20[:, 1], -Z_2_soc20[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc20[:, 1] + 1j * Z_2_soc20[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc20[:, 1] + 1j * Z_2_soc20[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc20[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc20_highPrecision.txt", Z_2_soc20, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2_soc10 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values
    model_DFN = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        ) 


    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": 2e9},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN_1 = sim_DFN_1.solve(t_eval=t, initial_soc=0.1)

    simulated_U_L_1 =  sol_DFN_1["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST

    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L_1) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L_1)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN_1["Time [s]"].entries + x[1]) + x[2] - simulated_U_L_1
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L_1)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2_soc10.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2_soc10 = np.array(Z_2_soc10)

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2_soc10[:, 1], -Z_2_soc10[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4)

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2_soc10[:, 1] + 1j * Z_2_soc10[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2_soc10[:, 1] + 1j * Z_2_soc10[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2_soc10[:, 0]  # Assuming first column is frequency

# # Create figure with two subplots
# fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# # Magnitude plot
# ax[0].semilogx(freq, magnitude, color='b', linewidth=1.5)
# ax[0].set_ylabel('Absolute Impedance (Ω)', fontsize=12)
# ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)
# ax[0].set_title('Bode Plot', fontsize=14)

# # Phase plot
# ax[1].semilogx(freq, phase, color='r', linewidth=1.5)
# ax[1].set_xlabel('Frequency (Hz)', fontsize=12)
# ax[1].set_ylabel('Phase (°)', fontsize=12)
# ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

# Tidy layout
plt.tight_layout()
plt.show()

In [ ]:
# save to txt
np.savetxt("Z_2GPa_soc10_highPrecision.txt", Z_2_soc10, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 10 MPa, SOC 1

In [ ]:
hydrostatic_stress = 10e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_10MPa_soc100 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                },
                                        )

    param10 = pybamm.ParameterValues("Ai2020") 
    param10.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param10["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param10, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]


    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_10MPa_soc100.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_10MPa_soc100 = np.array(Z_10MPa_soc100)

# save to txt
np.savetxt("Z_10MPa_soc100.txt", Z_10MPa_soc100)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")



### $\sigma_h$ = 100 MPa, SOC 1

In [ ]:
hydrostatic_stress = 1e8

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_100MPa_soc100 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                },
                                        )

    param100 = pybamm.ParameterValues("Ai2020") 
    param100.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param100["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param100, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]


    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_100MPa_soc100.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_100MPa_soc100 = np.array(Z_100MPa_soc100)

# save to txt
np.savetxt("Z_100MPa_soc100.txt", Z_100MPa_soc100)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")



### Vsi rezultati

In [ ]:
plt.plot(Z_0_soc100[:, 1], -Z_0_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0', label="$\sigma_{\mathrm{h}}=0$ Pa") #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 1")
plt.plot(Z_0_soc90[:, 1], -Z_0_soc90[:, 2]+0.02, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.9")
plt.plot(Z_0_soc80[:, 1], -Z_0_soc80[:, 2]+0.04, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.8")
plt.plot(Z_0_soc70[:, 1], -Z_0_soc70[:, 2]+0.06, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.7")
plt.plot(Z_0_soc60[:, 1], -Z_0_soc60[:, 2]+0.08, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.6")
plt.plot(Z_0_soc50[:, 1], -Z_0_soc50[:, 2]+0.10, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.5")
plt.plot(Z_0_soc40[:, 1], -Z_0_soc40[:, 2]+0.12, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.4")
plt.plot(Z_0_soc30[:, 1], -Z_0_soc30[:, 2]+0.14, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.3")
plt.plot(Z_0_soc20[:, 1], -Z_0_soc20[:, 2]+0.16, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.2")
plt.plot(Z_0_soc10[:, 1], -Z_0_soc10[:, 2]+0.18, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C0') #label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.1")

plt.plot(Z_1_soc100[:, 1], -Z_1_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1', label="$\sigma_{\mathrm{h}}=1$e9 Pa") #label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 1")
plt.plot(Z_1_soc90[:, 1], -Z_1_soc90[:, 2]+0.02, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.9")
plt.plot(Z_1_soc80[:, 1], -Z_1_soc80[:, 2]+0.04, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.8")
plt.plot(Z_1_soc70[:, 1], -Z_1_soc70[:, 2]+0.06, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.7")
plt.plot(Z_1_soc60[:, 1], -Z_1_soc60[:, 2]+0.08, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.6")
plt.plot(Z_1_soc50[:, 1], -Z_1_soc50[:, 2]+0.10, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.5")
plt.plot(Z_1_soc40[:, 1], -Z_1_soc40[:, 2]+0.12, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.4")
plt.plot(Z_1_soc30[:, 1], -Z_1_soc30[:, 2]+0.14, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.3")
plt.plot(Z_1_soc20[:, 1], -Z_1_soc20[:, 2]+0.16, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.2")
plt.plot(Z_1_soc10[:, 1], -Z_1_soc10[:, 2]+0.18, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C1')#label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.1")

plt.plot(Z_2_soc100[:, 1], -Z_2_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2', label="$\sigma_{\mathrm{h}}=2$e9 Pa")  #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 1")
plt.plot(Z_2_soc90[:, 1], -Z_2_soc90[:, 2]+0.02, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.9")
plt.plot(Z_2_soc80[:, 1], -Z_2_soc80[:, 2]+0.04, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.8")
plt.plot(Z_2_soc70[:, 1], -Z_2_soc70[:, 2]+0.06, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.7")
plt.plot(Z_2_soc60[:, 1], -Z_2_soc60[:, 2]+0.08, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.6")
plt.plot(Z_2_soc50[:, 1], -Z_2_soc50[:, 2]+0.10, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.5")
plt.plot(Z_2_soc40[:, 1], -Z_2_soc40[:, 2]+0.12, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.4")
plt.plot(Z_2_soc30[:, 1], -Z_2_soc30[:, 2]+0.14, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.3")
plt.plot(Z_2_soc20[:, 1], -Z_2_soc20[:, 2]+0.16, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.2")
plt.plot(Z_2_soc10[:, 1], -Z_2_soc10[:, 2]+0.18, marker='o', linestyle='-', linewidth=1.5, markersize=4, color = 'C2') #label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.1")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.xlim(0, 0.23)
plt.ylim(0, 0.23)

plt.legend(loc = "center right", bbox_to_anchor=(1.6, 0.5))

In [ ]:
plt.plot(Z_0_soc100[:, 1], -Z_0_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 1")
plt.plot(Z_0_soc90[:, 1], -Z_0_soc90[:, 2]+0.02, marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.9")
plt.plot(Z_0_soc80[:, 1], -Z_0_soc80[:, 2]+0.04, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.8")
plt.plot(Z_0_soc70[:, 1], -Z_0_soc70[:, 2]+0.06, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.7")
plt.plot(Z_0_soc60[:, 1], -Z_0_soc60[:, 2]+0.08, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.6")
plt.plot(Z_0_soc50[:, 1], -Z_0_soc50[:, 2]+0.10, marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.5")
plt.plot(Z_0_soc40[:, 1], -Z_0_soc40[:, 2]+0.12, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.4")
plt.plot(Z_0_soc30[:, 1], -Z_0_soc30[:, 2]+0.14, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.3")
plt.plot(Z_0_soc20[:, 1], -Z_0_soc20[:, 2]+0.16, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.2")
plt.plot(Z_0_soc10[:, 1], -Z_0_soc10[:, 2]+0.18, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.1")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot ($\sigma_{\mathrm{h}}=0$ Pa)')
plt.xlim(0, 0.23)
plt.ylim(0, 0.23)

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_1_soc100[:, 1], -Z_1_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 1")
plt.plot(Z_1_soc90[:, 1], -Z_1_soc90[:, 2]+0.02, marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.9")
plt.plot(Z_1_soc80[:, 1], -Z_1_soc80[:, 2]+0.04, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.8")
plt.plot(Z_1_soc70[:, 1], -Z_1_soc70[:, 2]+0.06, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.7")
plt.plot(Z_1_soc60[:, 1], -Z_1_soc60[:, 2]+0.08, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.6")
plt.plot(Z_1_soc50[:, 1], -Z_1_soc50[:, 2]+0.1, marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.5")
plt.plot(Z_1_soc40[:, 1], -Z_1_soc40[:, 2]+0.12, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.4")
plt.plot(Z_1_soc30[:, 1], -Z_1_soc30[:, 2]+0.14, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.3")
plt.plot(Z_1_soc20[:, 1], -Z_1_soc20[:, 2]+0.16, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.2")
plt.plot(Z_1_soc10[:, 1], -Z_1_soc10[:, 2]+0.18, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.1")



# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot ($\sigma_{\mathrm{h}}=1$ GPa)')

plt.xlim(0, 0.23)
plt.ylim(0, 0.23)

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_2_soc100[:, 1], -Z_2_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 1")
plt.plot(Z_2_soc90[:, 1], -Z_2_soc90[:, 2]+0.02, marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.9")
plt.plot(Z_2_soc80[:, 1], -Z_2_soc80[:, 2]+0.04, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.8")
plt.plot(Z_2_soc70[:, 1], -Z_2_soc70[:, 2]+0.06, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.7")
plt.plot(Z_2_soc60[:, 1], -Z_2_soc60[:, 2]+0.08, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.6")
plt.plot(Z_2_soc50[:, 1], -Z_2_soc50[:, 2]+0.1, marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.5")
plt.plot(Z_2_soc40[:, 1], -Z_2_soc40[:, 2]+0.12, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.4")
plt.plot(Z_2_soc30[:, 1], -Z_2_soc30[:, 2]+0.14, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.3")
plt.plot(Z_2_soc20[:, 1], -Z_2_soc20[:, 2]+0.16, marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.2")
plt.plot(Z_2_soc10[:, 1], -Z_2_soc10[:, 2]+0.18, marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.1")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot ($\sigma_{\mathrm{h}}=2$ GPa)')

plt.xlim(0, 0.23)
plt.ylim(0, 0.23)

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
# import from txt
Z_0_soc100 = np.loadtxt("Z_0GPa_soc100_highPrecision.txt", skiprows=1)
Z_1_soc100 = np.loadtxt("Z_1GPa_soc100_highPrecision.txt", skiprows=1)
Z_2_soc100 = np.loadtxt("Z_2GPa_soc100_highPrecision.txt", skiprows=1)
Z_10MPa_soc100 = np.loadtxt("Z_10MPa_soc100.txt", skiprows=1)

In [ ]:
plt.plot(Z_0_soc100[:, 1], -Z_0_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 1")
# plt.plot(Z_1_soc100[:, 1], -Z_1_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 1")
# plt.plot(Z_2_soc100[:, 1], -Z_2_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 1")
plt.plot(Z_10MPa_soc100[:, 1], -Z_10MPa_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=10$ MPa, SOC = 1")
plt.plot(Z_100MPa_soc100[:, 1], -Z_100MPa_soc100[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=100$ MPa, SOC = 1")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc90[:, 1], -Z_0_soc90[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.9")
plt.plot(Z_1_soc90[:, 1], -Z_1_soc90[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.9")
plt.plot(Z_2_soc90[:, 1], -Z_2_soc90[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.9")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc80[:, 1], -Z_0_soc80[:, 2], marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.8")
plt.plot(Z_1_soc80[:, 1], -Z_1_soc80[:, 2], marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.8")
plt.plot(Z_2_soc80[:, 1], -Z_2_soc80[:, 2], marker='o', linestyle=':', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.8")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc70[:, 1], -Z_0_soc70[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.7")
plt.plot(Z_1_soc70[:, 1], -Z_1_soc70[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.7")
plt.plot(Z_2_soc70[:, 1], -Z_2_soc70[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.7")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc60[:, 1], -Z_0_soc60[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.6")
plt.plot(Z_1_soc60[:, 1], -Z_1_soc60[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.6")
plt.plot(Z_2_soc60[:, 1], -Z_2_soc60[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.6")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc50[:, 1], -Z_0_soc50[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.5")
plt.plot(Z_1_soc50[:, 1], -Z_1_soc50[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.5")
plt.plot(Z_2_soc50[:, 1], -Z_2_soc50[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.5")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc40[:, 1], -Z_0_soc40[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.4")
plt.plot(Z_1_soc40[:, 1], -Z_1_soc40[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.4")
plt.plot(Z_2_soc40[:, 1], -Z_2_soc40[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.4")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc30[:, 1], -Z_0_soc30[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.3")
plt.plot(Z_1_soc30[:, 1], -Z_1_soc30[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.3")
plt.plot(Z_2_soc30[:, 1], -Z_2_soc30[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.3")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc20[:, 1], -Z_0_soc20[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.2")
plt.plot(Z_1_soc20[:, 1], -Z_1_soc20[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.2")
plt.plot(Z_2_soc20[:, 1], -Z_2_soc20[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.2")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

In [ ]:
plt.plot(Z_0_soc10[:, 1], -Z_0_soc10[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=0$ Pa, SOC = 0.1")
plt.plot(Z_1_soc10[:, 1], -Z_1_soc10[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=1$e9 Pa, SOC = 0.1")
plt.plot(Z_2_soc10[:, 1], -Z_2_soc10[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, label="$\sigma_{\mathrm{h}}=2$e9 Pa, SOC = 0.1")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc = "center right", bbox_to_anchor=(1.9, 0.5))

# Različne $\Omega$

In [ ]:
param0 = pybamm.ParameterValues("Ai2020") 
param0.search("molar volume")

### $\Omega$ = 3.1e-6 $m^3/mol$ (default)

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omgDef = np.array(Z_0MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_0MPa_soc100_OmgDef.txt", Z_0MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0MPa_soc100_omgDef[:, 1], -Z_0MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0MPa_soc100_omgDef[:, 1] + 1j * Z_0MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0MPa_soc100_omgDef[:, 1] + 1j * Z_0MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": 1e6},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omgDef = np.array(Z_1MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_1MPa_soc100_OmgDef.txt", Z_1MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1MPa_soc100_omgDef[:, 1], -Z_1MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1MPa_soc100_omgDef[:, 1] + 1j * Z_1MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1MPa_soc100_omgDef[:, 1] + 1j * Z_1MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": 0.5e6},
        check_already_exists=False  
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omgDef = np.array(Z_05MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_05MPa_soc100_OmgDef.txt", Z_05MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_05MPa_soc100_omgDef[:, 1], -Z_05MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_05MPa_soc100_omgDef[:, 1] + 1j * Z_05MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
phase = np.angle(Z_05MPa_soc100_omgDef[:, 1] + 1j * Z_05MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_05MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omgDef = np.array(Z_01MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_01MPa_soc100_OmgDef.txt", Z_01MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_01MPa_soc100_omgDef[:, 1], -Z_01MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_01MPa_soc100_omgDef[:, 1] + 1j * Z_01MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
phase = np.angle(Z_01MPa_soc100_omgDef[:, 1] + 1j * Z_01MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_01MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_omgDef = np.array(Z_2MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_2MPa_soc100_OmgDef.txt", Z_2MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2MPa_soc100_omgDef[:, 1], -Z_2MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2MPa_soc100_omgDef[:, 1] + 1j * Z_2MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2MPa_soc100_omgDef[:, 1] + 1j * Z_2MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omgDef[:, 1], -Z_0MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_01MPa_soc100_omgDef[:, 1], -Z_01MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_05MPa_soc100_omgDef[:, 1], -Z_05MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_omgDef[:, 1], -Z_1MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_2MPa_soc100_omgDef[:, 1], -Z_2MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

### $\Omega$ = 1e-6 $m^3/mol$

#### $\sigma_h$ = 0 Pa, SOC = 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omg1 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 1e-6},
    check_already_exists=False
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omg1.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omg1 = np.array(Z_0MPa_soc100_omg1)

# save to txt
np.savetxt("Z_0MPa_soc100_Omg1e-6.txt", Z_0MPa_soc100_omg1)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0MPa_soc100_omg1[:, 1], -Z_0MPa_soc100_omg1[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label="$\sigma_{\mathrm{h}}=0$ MPa, SOC = 1, $\Omega$ = 1e-6 m3/mol")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0MPa_soc100_omg1[:, 1] + 1j * Z_0MPa_soc100_omg1[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0MPa_soc100_omg1[:, 1] + 1j * Z_0MPa_soc100_omg1[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0MPa_soc100_omg1[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 1 MPa, SOC = 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omg1 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    # PARAMETRI
    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": 1e6},
        check_already_exists=False  
    )
    param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 1e-6},
    check_already_exists=False
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omg1.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omg1 = np.array(Z_1MPa_soc100_omg1)

# save to txt
np.savetxt("Z_1MPa_soc100_Omg1e-6.txt", Z_1MPa_soc100_omg1)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1MPa_soc100_omg1[:, 1], -Z_1MPa_soc100_omg1[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label="$\sigma_{\mathrm{h}}=1$ MPa, SOC = 1, $\Omega$ = 1e-6 m3/mol")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1MPa_soc100_omg1[:, 1] + 1j * Z_1MPa_soc100_omg1[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1MPa_soc100_omg1[:, 1] + 1j * Z_1MPa_soc100_omg1[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1MPa_soc100_omg1[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()


#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omg1[:, 1], -Z_0MPa_soc100_omg1[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")
         
plt.plot(Z_1MPa_soc100_omg1[:, 1], -Z_1MPa_soc100_omg1[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

### $\Omega$ = 6.2e-6 $m^3/mol$ (2 x default)

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
hydrostatic_stress = 0
omega = 6.2e-6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omg6_2 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omg6_2.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omg6_2 = np.array(Z_0MPa_soc100_omg6_2)

# save to txt
np.savetxt("Z_0MPa_soc100_Omg6_2.txt", Z_0MPa_soc100_omg6_2)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0MPa_soc100_omg6_2[:, 1], -Z_0MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0MPa_soc100_omg6_2[:, 1] + 1j * Z_0MPa_soc100_omg6_2[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0MPa_soc100_omg6_2[:, 1] + 1j * Z_0MPa_soc100_omg6_2[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0MPa_soc100_omg6_2[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
hydrostic_stress = 1e6
omega = 6.2e-6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omg6_2 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostic_stress},
        check_already_exists=False  
    )
    param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omg6_2.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omg6_2 = np.array(Z_1MPa_soc100_omg6_2)

# save to txt
np.savetxt("Z_1MPa_soc100_Omg6_2.txt", Z_1MPa_soc100_omg6_2)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1MPa_soc100_omg6_2[:, 1], -Z_1MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1MPa_soc100_omg6_2[:, 1] + 1j * Z_1MPa_soc100_omg6_2[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1MPa_soc100_omg6_2[:, 1] + 1j * Z_1MPa_soc100_omg6_2[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1MPa_soc100_omg6_2[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.5e6
omega = 6.2e-6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omg6_2 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param05.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omg6_2.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omg6_2 = np.array(Z_05MPa_soc100_omg6_2)

# save to txt
np.savetxt("Z_05MPa_soc100_Omg6_2.txt", Z_05MPa_soc100_omg6_2)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_05MPa_soc100_omg6_2[:, 1], -Z_05MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_05MPa_soc100_omg6_2[:, 1] + 1j * Z_05MPa_soc100_omg6_2[:, 2])  # Combine real and imag parts
phase = np.angle(Z_05MPa_soc100_omg6_2[:, 1] + 1j * Z_05MPa_soc100_omg6_2[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_05MPa_soc100_omg6_2[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6
omega = 6.2e-6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omg6_2 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omg6_2.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omg6_2 = np.array(Z_01MPa_soc100_omg6_2)

# save to txt
np.savetxt("Z_01MPa_soc100_Omg6_2.txt", Z_01MPa_soc100_omg6_2)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_01MPa_soc100_omg6_2[:, 1], -Z_01MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_01MPa_soc100_omg6_2[:, 1] + 1j * Z_01MPa_soc100_omg6_2[:, 2])  # Combine real and imag parts
phase = np.angle(Z_01MPa_soc100_omg6_2[:, 1] + 1j * Z_01MPa_soc100_omg6_2[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_01MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6
omega = 6.2e-6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_omg6_2 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_omg6_2.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_omg6_2 = np.array(Z_2MPa_soc100_omg6_2)

# save to txt
np.savetxt("Z_2MPa_soc100_Omg6_2.txt", Z_2MPa_soc100_omg6_2)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2MPa_soc100_omg6_2[:, 1], -Z_2MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2MPa_soc100_omg6_2[:, 1] + 1j * Z_2MPa_soc100_omg6_2[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2MPa_soc100_omg6_2[:, 1] + 1j * Z_2MPa_soc100_omg6_2[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2MPa_soc100_omg6_2[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omg6_2[:, 1], -Z_0MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_01MPa_soc100_omg6_2[:, 1], -Z_01MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_05MPa_soc100_omg6_2[:, 1], -Z_05MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_omg6_2[:, 1], -Z_1MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_2MPa_soc100_omg6_2[:, 1], -Z_2MPa_soc100_omg6_2[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

### $\Omega$ = 1e-5 $m^3/mol$

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
hydrostatic_stress = 0
omega = 1e-5

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omg1e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omg1e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omg1e5 = np.array(Z_0MPa_soc100_omg1e5)

# save to txt
np.savetxt("Z_0MPa_soc100_Omg1e5.txt", Z_0MPa_soc100_omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0MPa_soc100_omg1e5[:, 1], -Z_0MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0MPa_soc100_omg1e5[:, 1] + 1j * Z_0MPa_soc100_omg1e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0MPa_soc100_omg1e5[:, 1] + 1j * Z_0MPa_soc100_omg1e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0MPa_soc100_omg1e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
hydrostic_stress = 1e6
omega = 1e-5

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omg1e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostic_stress},
        check_already_exists=False  
    )
    param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omg1e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omg1e5 = np.array(Z_1MPa_soc100_omg1e5)

# save to txt
np.savetxt("Z_1MPa_soc100_Omg1e5.txt", Z_1MPa_soc100_omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1MPa_soc100_omg1e5[:, 1], -Z_1MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1MPa_soc100_omg1e5[:, 1] + 1j * Z_1MPa_soc100_omg1e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1MPa_soc100_omg1e5[:, 1] + 1j * Z_1MPa_soc100_omg1e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1MPa_soc100_omg1e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.5e6
omega = 1e-5

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omg1e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param05.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omg1e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omg1e5 = np.array(Z_05MPa_soc100_omg1e5)

# save to txt
np.savetxt("Z_05MPa_soc100_Omg1e5.txt", Z_05MPa_soc100_omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_05MPa_soc100_omg1e5[:, 1], -Z_05MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_05MPa_soc100_omg1e5[:, 1] + 1j * Z_05MPa_soc100_omg1e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_05MPa_soc100_omg1e5[:, 1] + 1j * Z_05MPa_soc100_omg1e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_05MPa_soc100_omg1e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6
omega = 1e-5

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omg1e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omg1e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omg1e5 = np.array(Z_01MPa_soc100_omg1e5)

# save to txt
np.savetxt("Z_01MPa_soc100_Omg1e5.txt", Z_01MPa_soc100_omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_01MPa_soc100_omg1e5[:, 1], -Z_01MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_01MPa_soc100_omg1e5[:, 1] + 1j * Z_01MPa_soc100_omg1e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_01MPa_soc100_omg1e5[:, 1] + 1j * Z_01MPa_soc100_omg1e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_01MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6
omega = 1e-5

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_omg1e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_omg1e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_omg1e5 = np.array(Z_2MPa_soc100_omg1e5)

# save to txt
np.savetxt("Z_2MPa_soc100_Omg1e5.txt", Z_2MPa_soc100_omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2MPa_soc100_omg1e5[:, 1], -Z_2MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2MPa_soc100_omg1e5[:, 1] + 1j * Z_2MPa_soc100_omg1e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2MPa_soc100_omg1e5[:, 1] + 1j * Z_2MPa_soc100_omg1e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2MPa_soc100_omg1e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omg1e5[:, 1], -Z_0MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_01MPa_soc100_omg1e5[:, 1], -Z_01MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_05MPa_soc100_omg1e5[:, 1], -Z_05MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_omg1e5[:, 1], -Z_1MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_2MPa_soc100_omg1e5[:, 1], -Z_2MPa_soc100_omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

### $\Omega$ = 3.2e-5 $m^3/mol$

In [ ]:
omega = 3.2e-5

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
hydrostatic_stress = 0

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omg3_2e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omg3_2e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omg3_2e5 = np.array(Z_0MPa_soc100_omg3_2e5)

# save to txt
np.savetxt("Z_0MPa_soc100_omg3_2e5.txt", Z_0MPa_soc100_omg3_2e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0MPa_soc100_omg3_2e5[:, 1], -Z_0MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0MPa_soc100_omg3_2e5[:, 1] + 1j * Z_0MPa_soc100_omg3_2e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0MPa_soc100_omg3_2e5[:, 1] + 1j * Z_0MPa_soc100_omg3_2e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0MPa_soc100_omg3_2e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
hydrostic_stress = 1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omg3_2e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostic_stress},
        check_already_exists=False  
    )
    param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omg3_2e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omg3_2e5 = np.array(Z_1MPa_soc100_omg3_2e5)

# save to txt
np.savetxt("Z_1MPa_soc100_Omg3_2e5.txt", Z_1MPa_soc100_omg3_2e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1MPa_soc100_omg3_2e5[:, 1], -Z_1MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1MPa_soc100_omg3_2e5[:, 1] + 1j * Z_1MPa_soc100_omg3_2e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1MPa_soc100_omg3_2e5[:, 1] + 1j * Z_1MPa_soc100_omg3_2e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1MPa_soc100_omg3_2e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.5e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omg3_2e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param05.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omg3_2e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omg3_2e5 = np.array(Z_05MPa_soc100_omg3_2e5)

# save to txt
np.savetxt("Z_05MPa_soc100_Omg3_2e5.txt", Z_05MPa_soc100_omg3_2e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_05MPa_soc100_omg3_2e5[:, 1], -Z_05MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_05MPa_soc100_omg3_2e5[:, 1] + 1j * Z_05MPa_soc100_omg3_2e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_05MPa_soc100_omg3_2e5[:, 1] + 1j * Z_05MPa_soc100_omg3_2e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_05MPa_soc100_omg3_2e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omg3_2e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omg3_2e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omg3_2e5 = np.array(Z_01MPa_soc100_omg3_2e5)

# save to txt
np.savetxt("Z_01MPa_soc100_Omg3_2e5.txt", Z_01MPa_soc100_omg3_2e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_01MPa_soc100_omg3_2e5[:, 1], -Z_01MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_01MPa_soc100_omg3_2e5[:, 1] + 1j * Z_01MPa_soc100_omg3_2e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_01MPa_soc100_omg3_2e5[:, 1] + 1j * Z_01MPa_soc100_omg3_2e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_01MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_omg3_2e5 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_omg3_2e5.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_omg3_2e5 = np.array(Z_2MPa_soc100_omg3_2e5)

# save to txt
np.savetxt("Z_2MPa_soc100_Omg3_2e5.txt", Z_2MPa_soc100_omg3_2e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2MPa_soc100_omg3_2e5[:, 1], -Z_2MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2MPa_soc100_omg3_2e5[:, 1] + 1j * Z_2MPa_soc100_omg3_2e5[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2MPa_soc100_omg3_2e5[:, 1] + 1j * Z_2MPa_soc100_omg3_2e5[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2MPa_soc100_omg3_2e5[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omg3_2e5[:, 1], -Z_0MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_01MPa_soc100_omg3_2e5[:, 1], -Z_01MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_05MPa_soc100_omg3_2e5[:, 1], -Z_05MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_omg3_2e5[:, 1], -Z_1MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_2MPa_soc100_omg3_2e5[:, 1], -Z_2MPa_soc100_omg3_2e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

### $\Omega$ = 3.2e-4 $m^3/mol$

In [ ]:
omega = 3.2e-4

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
hydrostatic_stress = 0

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omg3_2e4 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omg3_2e4.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omg3_2e4 = np.array(Z_0MPa_soc100_omg3_2e4)

# save to txt
np.savetxt("Z_0MPa_soc100_omg3_2e4.txt", Z_0MPa_soc100_omg3_2e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_0MPa_soc100_omg3_2e4[:, 1], -Z_0MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_0MPa_soc100_omg3_2e4[:, 1] + 1j * Z_0MPa_soc100_omg3_2e4[:, 2])  # Combine real and imag parts
phase = np.angle(Z_0MPa_soc100_omg3_2e4[:, 1] + 1j * Z_0MPa_soc100_omg3_2e4[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_0MPa_soc100_omg3_2e4[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
hydrostic_stress = 1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omg3_2e4 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostic_stress},
        check_already_exists=False  
    )
    param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omg3_2e4.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omg3_2e4 = np.array(Z_1MPa_soc100_omg3_2e4)

# save to txt
np.savetxt("Z_1MPa_soc100_Omg3_2e4.txt", Z_1MPa_soc100_omg3_2e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_1MPa_soc100_omg3_2e4[:, 1], -Z_1MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_1MPa_soc100_omg3_2e4[:, 1] + 1j * Z_1MPa_soc100_omg3_2e4[:, 2])  # Combine real and imag parts
phase = np.angle(Z_1MPa_soc100_omg3_2e4[:, 1] + 1j * Z_1MPa_soc100_omg3_2e4[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_1MPa_soc100_omg3_2e4[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.5e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omg3_2e4 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param05.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omg3_2e4.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omg3_2e4 = np.array(Z_05MPa_soc100_omg3_2e4)

# save to txt
np.savetxt("Z_05MPa_soc100_Omg3_2e4.txt", Z_05MPa_soc100_omg3_2e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_05MPa_soc100_omg3_2e4[:, 1], -Z_05MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_05MPa_soc100_omg3_2e4[:, 1] + 1j * Z_05MPa_soc100_omg3_2e4[:, 2])  # Combine real and imag parts
phase = np.angle(Z_05MPa_soc100_omg3_2e4[:, 1] + 1j * Z_05MPa_soc100_omg3_2e4[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_05MPa_soc100_omg3_2e4[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omg3_2e4 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omg3_2e4.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omg3_2e4 = np.array(Z_01MPa_soc100_omg3_2e4)

# save to txt
np.savetxt("Z_01MPa_soc100_Omg3_2e4.txt", Z_01MPa_soc100_omg3_2e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_01MPa_soc100_omg3_2e4[:, 1], -Z_01MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_01MPa_soc100_omg3_2e4[:, 1] + 1j * Z_01MPa_soc100_omg3_2e4[:, 2])  # Combine real and imag parts
phase = np.angle(Z_01MPa_soc100_omg3_2e4[:, 1] + 1j * Z_01MPa_soc100_omg3_2e4[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_01MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_omg3_2e4 = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2.update(
    {"Negative electrode partial molar volume [m3.mol-1]": omega},
    check_already_exists=False
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_omg3_2e4.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_omg3_2e4 = np.array(Z_2MPa_soc100_omg3_2e4)

# save to txt
np.savetxt("Z_2MPa_soc100_Omg3_2e4.txt", Z_2MPa_soc100_omg3_2e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#NARIŠEMO NIQUISTOV DIAGRAM
plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
plt.plot(Z_2MPa_soc100_omg3_2e4[:, 1], -Z_2MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Add grid for better readability
plt.grid(True, linestyle='--', linewidth=0.5)

# Equal aspect ratio for true representation of impedance
plt.axis('equal')

# Optional: add minor ticks
plt.minorticks_on()
plt.tick_params(which='both', direction='in', top=True, right=True)

plt.tight_layout()
plt.show()


# Compute magnitude (absolute value) and phase (angle)
magnitude = np.abs(Z_2MPa_soc100_omg3_2e4[:, 1] + 1j * Z_2MPa_soc100_omg3_2e4[:, 2])  # Combine real and imag parts
phase = np.angle(Z_2MPa_soc100_omg3_2e4[:, 1] + 1j * Z_2MPa_soc100_omg3_2e4[:, 2], deg=True)  # Phase in degrees

# Frequency axis (log scale typical for Bode)
freq = Z_2MPa_soc100_omg3_2e4[:, 0]  # Assuming first column is frequency

# Tidy layout
plt.tight_layout()
plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omg3_2e4[:, 1], -Z_0MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_01MPa_soc100_omg3_2e4[:, 1], -Z_01MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_05MPa_soc100_omg3_2e4[:, 1], -Z_05MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_omg3_2e4[:, 1], -Z_1MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_2MPa_soc100_omg3_2e4[:, 1], -Z_2MPa_soc100_omg3_2e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Večji spekter frekvenc

### $\Omega$ = 3.1e-6 $m^3/mol$ (default)

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_omgDef_vecF = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_omgDef_vecF.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_omgDef_vecF = np.array(Z_0MPa_soc100_omgDef_vecF)

# save to txt
np.savetxt("Z_0MPa_soc100_OmgDef_vecF.txt", Z_0MPa_soc100_omgDef_vecF)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_0MPa_soc100_omgDef_vecF[:, 1], -Z_0MPa_soc100_omgDef_vecF[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_0MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_0MPa_soc100_omgDef_vecF[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_0MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_0MPa_soc100_omgDef_vecF[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_0MPa_soc100_omgDef_vecF[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_omgDef_vecF = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": 1e6},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_omgDef_vecF.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_omgDef_vecF = np.array(Z_1MPa_soc100_omgDef_vecF)

# save to txt
np.savetxt("Z_1MPa_soc100_OmgDef_vecF.txt", Z_1MPa_soc100_omgDef_vecF)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_1MPa_soc100_omgDef_vecF[:, 1], -Z_1MPa_soc100_omgDef_vecF[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_1MPa_soc100_omgDef_vecF[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_1MPa_soc100_omgDef_vecF[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1MPa_soc100_omgDef_vecF[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omgDef_vecF = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": 0.5e6},
        check_already_exists=False  
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omgDef_vecF.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omgDef_vecF = np.array(Z_05MPa_soc100_omgDef_vecF)

# save to txt
np.savetxt("Z_05MPa_soc100_OmgDef_vecF.txt", Z_05MPa_soc100_omgDef_vecF)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_05MPa_soc100_omgDef_vecF[:, 1], -Z_05MPa_soc100_omgDef_vecF[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_05MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_05MPa_soc100_omgDef_vecF[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_05MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_05MPa_soc100_omgDef_vecF[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_05MPa_soc100_omgDef_vecF[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omgDef_vecF = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omgDef_vecF.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omgDef_vecF = np.array(Z_01MPa_soc100_omgDef_vecF)

# save to txt
np.savetxt("Z_01MPa_soc100_OmgDef_vecF.txt", Z_01MPa_soc100_omgDef_vecF)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_01MPa_soc100_omgDef_vecF[:, 1], -Z_01MPa_soc100_omgDef_vecF[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_01MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_01MPa_soc100_omgDef_vecF[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_01MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_01MPa_soc100_omgDef_vecF[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_01MPa_soc100_omgDef_vecF[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_omgDef_vecF = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_omgDef_vecF.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_omgDef_vecF = np.array(Z_2MPa_soc100_omgDef_vecF)

# save to txt
np.savetxt("Z_2MPa_soc100_OmgDef_vecF.txt", Z_2MPa_soc100_omgDef_vecF)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_2MPa_soc100_omgDef_vecF[:, 1], -Z_2MPa_soc100_omgDef_vecF[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_2MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_2MPa_soc100_omgDef_vecF[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_2MPa_soc100_omgDef_vecF[:, 1] + 1j * Z_2MPa_soc100_omgDef_vecF[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_2MPa_soc100_omgDef_vecF[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_omgDef[:, 1], -Z_0MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_01MPa_soc100_omgDef[:, 1], -Z_01MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_05MPa_soc100_omgDef[:, 1], -Z_05MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_omgDef[:, 1], -Z_1MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_2MPa_soc100_omgDef[:, 1], -Z_2MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# Vključimo SEI

V člankih je velikokrat navedeno, da se impedanca poveča zaradi kompresije SEI.

### $\Omega$ = 3.1e-6 $m^3/mol$ (default)

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_SEI = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                "SEI": "solvent-diffusion limited",
                                                "SEI porosity change": "true",},
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_SEI.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_SEI = np.array(Z_0MPa_soc100_SEI)

# save to txt
np.savetxt("Z_0MPa_soc100_SEI.txt", Z_0MPa_soc100_SEI)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_0MPa_soc100_SEI[:, 1], -Z_0MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_0MPa_soc100_SEI[:, 1] + 1j * Z_0MPa_soc100_SEI[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_0MPa_soc100_SEI[:, 1] + 1j * Z_0MPa_soc100_SEI[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_0MPa_soc100_SEI[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_SEI = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                "SEI": "solvent-diffusion limited",
                                                "SEI porosity change": "true",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": 1e6},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_SEI.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_SEI = np.array(Z_1MPa_soc100_SEI)

# save to txt
np.savetxt("Z_1MPa_soc100_SEI.txt", Z_1MPa_soc100_SEI)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_1MPa_soc100_SEI[:, 1], -Z_1MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1MPa_soc100_SEI[:, 1] + 1j * Z_1MPa_soc100_SEI[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1MPa_soc100_SEI[:, 1] + 1j * Z_1MPa_soc100_SEI[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1MPa_soc100_SEI[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": 0.5e6},
        check_already_exists=False  
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_omgDef = np.array(Z_05MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_05MPa_soc100_OmgDef_vecF.txt", Z_05MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_05MPa_soc100_omgDef[:, 1], -Z_05MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_05MPa_soc100_omgDef[:, 1] + 1j * Z_05MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_05MPa_soc100_omgDef[:, 1] + 1j * Z_05MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_05MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_omgDef = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling only",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_omgDef.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_omgDef = np.array(Z_01MPa_soc100_omgDef)

# save to txt
np.savetxt("Z_01MPa_soc100_OmgDef_vecF.txt", Z_01MPa_soc100_omgDef)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_01MPa_soc100_omgDef[:, 1], -Z_01MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_01MPa_soc100_omgDef[:, 1] + 1j * Z_01MPa_soc100_omgDef[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_01MPa_soc100_omgDef[:, 1] + 1j * Z_01MPa_soc100_omgDef[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_01MPa_soc100_omgDef[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_SEI = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                "SEI": "solvent-diffusion limited",
                                                "SEI porosity change": "true",},
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_SEI.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_SEI = np.array(Z_2MPa_soc100_SEI)

# save to txt
np.savetxt("Z_2MPa_soc100_SEI.txt", Z_2MPa_soc100_SEI)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_2MPa_soc100_SEI[:, 1], -Z_2MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_2MPa_soc100_SEI[:, 1] + 1j * Z_2MPa_soc100_SEI[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_2MPa_soc100_SEI[:, 1] + 1j * Z_2MPa_soc100_SEI[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_2MPa_soc100_SEI[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_SEI[:, 1], -Z_0MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, SEI = on")

# plt.plot(Z_01MPa_soc100_SEI[:, 1], -Z_01MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#         label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# plt.plot(Z_05MPa_soc100_SEI[:, 1], -Z_05MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#         label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

plt.plot(Z_1MPa_soc100_SEI[:, 1], -Z_1MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}, SEI = on")

plt.plot(Z_2MPa_soc100_SEI[:, 1], -Z_2MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}, SEI = on")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

In [ ]:
plt.plot(Z_0MPa_soc100_omgDef[:, 1], -Z_0MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, SEI = off")


plt.plot(Z_0MPa_soc100_SEI[:, 1], -Z_0MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, SEI = on")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

#### $\sigma_h$ = 1 GPa, SOC 1

In [ ]:
hydrostatic_stress = 1e9

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_SEI = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                "SEI": "solvent-diffusion limited",
                                                "SEI porosity change": "true",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_SEI.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_SEI = np.array(Z_1MPa_soc100_SEI)

# save to txt
np.savetxt("Z_1MPa_soc100_SEI.txt", Z_1MPa_soc100_SEI)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_1MPa_soc100_SEI[:, 1], -Z_1MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1MPa_soc100_SEI[:, 1] + 1j * Z_1MPa_soc100_SEI[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1MPa_soc100_SEI[:, 1] + 1j * Z_1MPa_soc100_SEI[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1MPa_soc100_SEI[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

In [ ]:
plt.plot(Z_1MPa_soc100_SEI[:, 1], -Z_1MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, SEI = on")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

#### $\sigma_h$ = 2 GPa, SOC 1

In [ ]:
hydrostatic_stress = 2e9

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_SEI = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                "SEI": "solvent-diffusion limited",
                                                "SEI porosity change": "true",},
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_SEI.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_SEI = np.array(Z_1MPa_soc100_SEI)

# save to txt
np.savetxt("Z_2GPa_soc100_SEI.txt", Z_1MPa_soc100_SEI)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_1MPa_soc100_SEI[:, 1], -Z_1MPa_soc100_SEI[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1MPa_soc100_SEI[:, 1] + 1j * Z_1MPa_soc100_SEI[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1MPa_soc100_SEI[:, 1] + 1j * Z_1MPa_soc100_SEI[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1MPa_soc100_SEI[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

# Vključimo particle cracking

### $\Omega$ = 3.1e-6 $m^3/mol$ (default)

#### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0MPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling and cracking",
                                                },
                                        )

    param0 = pybamm.ParameterValues("Ai2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": 0},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0MPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0MPa_soc100_cracking = np.array(Z_0MPa_soc100_cracking)

# save to txt
np.savetxt("Z_0MPa_soc100_cracking.txt", Z_0MPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_0MPa_soc100_cracking[:, 1], -Z_0MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_0MPa_soc100_cracking[:, 1] + 1j * Z_0MPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_0MPa_soc100_cracking[:, 1] + 1j * Z_0MPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_0MPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 1 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1MPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling and cracking",
                                                },
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": 1e6},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1MPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1MPa_soc100_cracking = np.array(Z_1MPa_soc100_cracking)

# save to txt
np.savetxt("Z_1MPa_soc100_cracking.txt", Z_1MPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_1MPa_soc100_cracking[:, 1], -Z_1MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1MPa_soc100_cracking[:, 1] + 1j * Z_1MPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1MPa_soc100_cracking[:, 1] + 1j * Z_1MPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1MPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

In [ ]:
plt.plot(Z_0MPa_soc100_cracking[:, 1], -Z_0MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")

plt.plot(Z_1MPa_soc100_cracking[:, 1], -Z_1MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")


# Labels and title
plt.xlabel('Re[Z] (Ω)') 
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))


#### $\sigma_h$ = 0.5 MPa, SOC 1

In [ ]:
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_05MPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling and cracking",
                                                 },
                                        )

    param05 = pybamm.ParameterValues("Ai2020") 
    param05.update(
        {"Hydrostatic stress [Pa]": 0.5e6},
        check_already_exists=False  
    )
    param05["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param05, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_05MPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_05MPa_soc100_cracking = np.array(Z_05MPa_soc100_cracking)

# save to txt
np.savetxt("Z_05MPa_soc100_cracking.txt", Z_05MPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_05MPa_soc100_cracking[:, 1], -Z_05MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_05MPa_soc100_cracking[:, 1] + 1j * Z_05MPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_05MPa_soc100_cracking[:, 1] + 1j * Z_05MPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_05MPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 0.1 MPa, SOC 1

In [ ]:
hydrostatic_stress = 0.1e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_01MPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                  "particle": "Fickian diffusion",
                                                 "particle mechanics": "swelling and cracking",},
                                        )

    param01 = pybamm.ParameterValues("Ai2020") 
    param01.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param01["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param01, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_01MPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_01MPa_soc100_cracking = np.array(Z_01MPa_soc100_cracking)

# save to txt
np.savetxt("Z_01MPa_soc100_cracking.txt", Z_01MPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_01MPa_soc100_cracking[:, 1], -Z_01MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_01MPa_soc100_cracking[:, 1] + 1j * Z_01MPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_01MPa_soc100_cracking[:, 1] + 1j * Z_01MPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_01MPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 2 MPa, SOC 1

In [ ]:
hydrostatic_stress = 2e6

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2MPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling and cracking",
                                                },
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2MPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2MPa_soc100_cracking = np.array(Z_2MPa_soc100_cracking)

# save to txt
np.savetxt("Z_2MPa_soc100_cracking.txt", Z_2MPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_2MPa_soc100_cracking[:, 1], -Z_2MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_2MPa_soc100_cracking[:, 1] + 1j * Z_2MPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_2MPa_soc100_cracking[:, 1] + 1j * Z_2MPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_2MPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### Primerjava

In [ ]:
plt.plot(Z_0MPa_soc100_cracking[:, 1], -Z_0MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")

plt.plot(Z_01MPa_soc100_cracking[:, 1], -Z_01MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param01['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param01['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")

plt.plot(Z_05MPa_soc100_cracking[:, 1], -Z_05MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param05['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param05['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")

plt.plot(Z_1MPa_soc100_cracking[:, 1], -Z_1MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")

plt.plot(Z_2MPa_soc100_cracking[:, 1], -Z_2MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$  = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

In [ ]:
#import from txt file
Z_0MPa_soc100_omgDef = np.loadtxt("Z_0MPa_soc100_omgDef.txt")  

plt.plot(Z_0MPa_soc100_omgDef[:, 1], -Z_0MPa_soc100_omgDef[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = off")


plt.plot(Z_0MPa_soc100_cracking[:, 1], -Z_0MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = on")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')


plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

#### $\sigma_h$ = 1 GPa, SOC 1

In [ ]:
hydrostatic_stress = 1e9

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_1GPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling and cracking",
                                                },
                                        )

    param1 = pybamm.ParameterValues("Ai2020") 
    param1.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param1["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_1GPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_1GPa_soc100_cracking = np.array(Z_1GPa_soc100_cracking)

# save to txt
np.savetxt("Z_1GPa_soc100_cracking.txt", Z_1GPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_1GPa_soc100_cracking[:, 1], -Z_1GPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_1GPa_soc100_cracking[:, 1] + 1j * Z_1GPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_1GPa_soc100_cracking[:, 1] + 1j * Z_1GPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_1GPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

#### $\sigma_h$ = 2 GPa, SOC 1

In [ ]:
hydrostatic_stress = 2e9

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_2GPa_soc100_cracking = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):
    # model = pybamm.lithium_ion.DFN(options={"surface form": "differential"},)
    # model, ki ima le mehansko degradacijo
    # model = pybamm.lithium_ion.BasicDFN()
    # param = model.default_parameter_values 

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling and cracking",
                                                },
                                        )

    param2 = pybamm.ParameterValues("Ai2020") 
    param2.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param2["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #print(est_amp, est_mean, est_phase*180/np.pi) #JUST FOR DEBUGGING

    #plt.plot(est_amp * np.sin(2*np.pi*freq*t + est_phase) + est_mean)
    #plt.plot(simulated_U_L)
    #plt.show()

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_2GPa_soc100_cracking.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_2GPa_soc100_cracking = np.array(Z_2GPa_soc100_cracking)

# save to txt
np.savetxt("Z_2GPa_soc100_cracking.txt", Z_2GPa_soc100_cracking)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

# #NARIŠEMO NIQUISTOV DIAGRAM
# plt.figure(figsize=(6, 6))  # Make the figure square for Nyquist plots
# plt.plot(Z_2GPa_soc100_cracking[:, 1], -Z_2GPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
#          label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} MPa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}")

# # Labels and title
# plt.xlabel('Re[Z] (Ω)')
# plt.ylabel('-Im[Z] (Ω)')
# plt.title('Nyquist Plot')
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# # Add grid for better readability
# plt.grid(True, linestyle='--', linewidth=0.5)

# # Equal aspect ratio for true representation of impedance
# plt.axis('equal')

# # Optional: add minor ticks
# plt.minorticks_on()
# plt.tick_params(which='both', direction='in', top=True, right=True)

# plt.tight_layout()
# plt.show()


# # Compute magnitude (absolute value) and phase (angle)
# magnitude = np.abs(Z_2GPa_soc100_cracking[:, 1] + 1j * Z_2GPa_soc100_cracking[:, 2])  # Combine real and imag parts
# phase = np.angle(Z_2GPa_soc100_cracking[:, 1] + 1j * Z_2GPa_soc100_cracking[:, 2], deg=True)  # Phase in degrees

# # Frequency axis (log scale typical for Bode)
# freq = Z_2GPa_soc100_cracking[:, 0]  # Assuming first column is frequency

# # Tidy layout
# plt.tight_layout()
# plt.show()

In [ ]:
Z_0GPa_soc100_highPrecision = np.loadtxt("Z_0GPa_soc100_highPrecision.txt")
Z_1GPa_soc100_highPrecision = np.loadtxt("Z_1GPa_soc100_highPrecision.txt")
Z_2GPa_soc100_highPrecision = np.loadtxt("Z_2GPa_soc100_highPrecision.txt")

plt.plot(Z_0MPa_soc100_cracking[:, 1], -Z_0MPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = on")  

plt.plot(Z_1GPa_soc100_cracking[:, 1], -Z_1GPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = on")

plt.plot(Z_2GPa_soc100_cracking[:, 1], -Z_2GPa_soc100_cracking[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = on")

plt.plot(Z_0GPa_soc100_highPrecision[:, 1], -Z_0GPa_soc100_highPrecision[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param0['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = off")

plt.plot(Z_1GPa_soc100_highPrecision[:, 1], -Z_1GPa_soc100_highPrecision[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = off")       

plt.plot(Z_2GPa_soc100_highPrecision[:, 1], -Z_2GPa_soc100_highPrecision[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, $\Omega$ = {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e}, cracking = off")       

# Labels and title
plt.xlabel('Re[Z] (Ω)') 
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

# Half-cell model

## Positive half-cell

In [ ]:
hydrostatic_stress = 0

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 100000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0Pa_soc100_positiveHalf = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):

    model = pybamm.lithium_ion.DFN(options={"working electrode": "positive",
                                                "surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                # "particle mechanics": "swelling only", # odpravimo KeyError: "'Positive electrode partial molar volume [m3.mol-1]' not found. 
                                                },
                                        )

    param0 = pybamm.ParameterValues("Xu2019") 
    param0.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    ##################################################### DODALA zaradi KeyError-ja "'Negative electrode partial molar volume [m3.mol-1]' not found. 
    param0.update(
        {"Negative electrode partial molar volume [m3.mol-1]": param0['Lithium metal partial molar volume [m3.mol-1]']}, #setting partial molar volume to zero to avoid mechanics in the negative electrode
        check_already_exists=False
    )

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0Pa_soc100_positiveHalf.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0Pa_soc100_positiveHalf = np.array(Z_0Pa_soc100_positiveHalf)

# save to txt
np.savetxt("Z_0Pa_soc100_positiveHalf.txt", Z_0Pa_soc100_positiveHalf)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")



In [ ]:
plt.plot(Z_0Pa_soc100_positiveHalf[:, 1], -Z_0Pa_soc100_positiveHalf[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa, SOC = 1, positive half-cell")  

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot') 
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

# Symmetrical cell

Korekcije v PyBaMM mapi `src/input/parameters/lithium_ion`: 
1. izdelava `SymLCOAi2020` - simetrična celica s katodama  -> prepisala vse anodne parametre s katodnimi
2. izdelava `SymGraphiteAi2020` - simetrična celica z anodama -> prepisala vse katodne parametre z anodnimi
3. korekcija `__init__.py` -> dodala imeni obeh setov parametrov

Nato v glavni mapi PyBaMM:
4. korekcija dokumenta `pyproject.toml` -> dodala:
```
SymGraphiteAi2020 = "pybamm.input.parameters.lithium_ion.SymGraphiteAi2020:get_parameter_values"
SymLCOAi2020 = "pybamm.input.parameters.lithium_ion.SymLCOAi2020:get_parameter_values"
```

Potem sem zagnala `install_pybamm.ipynb`.

In [ ]:
# import
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

## LiCo2

### $\sigma_h$ = 0 Pa, SOC 1

In [ ]:
hydrostatic_stress = 0

I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z_0Pa_soc100_SymLCO = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):

    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                "particle mechanics": "swelling only",
                                                },
                                        )

    param0 = pybamm.ParameterValues("SymLCOAi2020") 
    param0.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param0["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN_0.solve(t_eval=t, initial_soc=1)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]


    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z_0Pa_soc100_SymLCO.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z_0Pa_soc100_SymLCO = np.array(Z_0Pa_soc100_SymLCO)

# save to txt
np.savetxt("Z_0Pa_soc100_SymLCO.txt", Z_0Pa_soc100_SymLCO)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")



In [ ]:
# param0.search("voltage")
# model.variables.search("voltage")

# EIS pri konst. napetosti

In [ ]:
# import
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0},
    check_already_exists=False  
)


param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i]
                                        )
# draw line at soc = 0.5
plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# draw line at 3.6925 V
plt.axhline(y=3.6925, color='gray', linestyle='--', label='V = 3.6925 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

## SOC = 0.5

### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################

# rename variables
Z_0Pa_soc50 = []
Z_0Pa_soc50 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_0Pa_soc50.pkl")

# save to txt
np.save("Z_0Pa_soc50", Z_0Pa_soc50)
np.savetxt("Z_0Pa_soc50.txt", Z_0Pa_soc50)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e8 Pa

In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1

#####################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

############################
# rename variables
Z_1e8Pa_soc50 = []
Z_1e8Pa_soc50 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e8Pa_soc50.pkl")

# save to txt
np.save("Z_1e8Pa_soc50", Z_1e8Pa_soc50)
np.savetxt("Z_1e8Pa_soc50.txt", Z_1e8Pa_soc50)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e9 Pa

In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################
# rename variables
Z_1e9Pa_soc50 = []
Z_1e9Pa_soc50 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e9Pa_soc50.pkl")

# save to txt
np.save("Z_1e9Pa_soc50", Z_1e9Pa_soc50)
np.savetxt("Z_1e9Pa_soc50.txt", Z_1e9Pa_soc50)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### Vsi rezultati

In [ ]:
Z_0Pa_soc50 = np.load("Z_0Pa_soc50.npy")
# Z_1e8Pa_soc50 = np.load("Z_1e8Pa_soc50.npy")
# Z_1e9Pa_soc50 = np.load("Z_1e9Pa_soc50.npy")

plt.plot(Z_0Pa_soc50[:, 1], -Z_0Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc50[:, 1], -Z_1e8Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e9Pa_soc50[:, 1], -Z_1e9Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("sol_0Pa_soc50.pkl")
sol_1e8Pa = pybamm.load("sol_1e8Pa_soc50.pkl")
sol_1e9Pa = pybamm.load("sol_1e9Pa_soc50.pkl")

plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")
plt.plot(sol_1e9Pa["Time [s]"].entries, sol_1e9Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")   

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

## SOC = 50 %, $\Omega$ = 3.1e-5 (10x)

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0},
    check_already_exists=False  
)
param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
    check_already_exists=False  
)


param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8},
    check_already_exists=False  
)
param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9},
    check_already_exists=False  
)
param2.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
    check_already_exists=False  
)

param1e7 = pybamm.ParameterValues("Ai2020")
param1e7.update(
    {"Hydrostatic stress [Pa]": 1e7},
    check_already_exists=False  
)
param1e7.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )
sim_DFN_1e7 = pybamm.Simulation(model= model, parameter_values=param1e7, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1e7 = sim_DFN_1e7.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1e7 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

Q_discharged = sol_DFN_1e7["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1e7 = (Q_discharged / Q_max)
SOC_1e7 = 1 - DOD_1e7

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2,
        SOC_1e7
         ]

y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data,
        sol_DFN_1e7["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1e7['Hydrostatic stress [Pa]']:.0e} Pa",
                                        ][i]
                                        )
# draw line at soc = 0.5
plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# draw line at 3.6925 V
plt.axhline(y=3.6925, color='gray', linestyle='--', label='V = 3.6925 V')
plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.title("$\Omega$ = 3.1e-5 m3.mol-1")


### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

In [ ]:
# target voltage
target_voltage_1e7 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1e7 = (np.abs(sol_DFN_1e7["Voltage [V]"].entries - target_voltage_1e7 )).argmin()
target_soc_1e7 = SOC_1e7[index_voltage_1e7]

print(f"For hydrostatic stress of {param1e7['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1e7['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1e7['Voltage [V]'].entries[index_voltage_1e7]:.4f} V at index {index_voltage_1e7}.")
print(f"Corresponding SOC: {target_soc_1e7:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################

# rename variables
Z_0Pa_soc50_Omg1e5 = []
Z_0Pa_soc50_Omg1e5 = Z
initialSOC0_Omg1e5 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_0Pa_soc50_Omg1e5.pkl")

# save to txt
np.save("Z_0Pa_soc50_Omg1e5", Z_0Pa_soc50_Omg1e5)
np.savetxt("Z_0Pa_soc50_Omg1e5.txt", Z_0Pa_soc50_Omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e8 Pa

In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1

#####################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
        check_already_exists=False
        )
    
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

############################
# rename variables
Z_1e8Pa_soc50_Omg1e5 = []
Z_1e8Pa_soc50_Omg1e5 = Z
initialSOC1_Omg1e5 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e8Pa_soc50_Omg1e5.pkl")

# save to txt
np.save("Z_1e8Pa_soc50_Omg1e5", Z_1e8Pa_soc50_Omg1e5)
np.savetxt("Z_1e8Pa_soc50_Omg1e5.txt", Z_1e8Pa_soc50_Omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e9 Pa

In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################
# rename variables
Z_1e9Pa_soc50_Omg1e5 = []
Z_1e9Pa_soc50_Omg1e5 = Z
initialSOC2_Omg1e5 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e9Pa_soc50_Omg1e5.pkl")

# save to txt
np.save("Z_1e9Pa_soc50_Omg1e5", Z_1e9Pa_soc50_Omg1e5)
np.savetxt("Z_1e9Pa_soc50_Omg1e5.txt", Z_1e9Pa_soc50_Omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_1e7

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-5},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################
# rename variables
Z_1e7Pa_soc50_Omg1e5 = []
Z_1e7Pa_soc50_Omg1e5 = Z
initialSOC_1e7_Omg1e5 = initialSOC
hydrostatic_stress_1e7 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e7Pa_soc50_Omg1e5.pkl")

# save to txt
np.save("Z_1e7Pa_soc50_Omg1e5", Z_1e7Pa_soc50_Omg1e5)
np.savetxt("Z_1e7Pa_soc50_Omg1e5.txt", Z_1e7Pa_soc50_Omg1e5)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### Vsi rezultati

In [ ]:
# Z_0Pa_soc50_Omg1e5 = np.load("Z_0Pa_soc50_Omg1e5.npy")
# Z_1e8Pa_soc50_Omg1e5 = np.load("Z_1e8Pa_soc50_Omg1e5.npy")
# Z_1e9Pa_soc50_Omg1e5 = np.load("Z_1e9Pa_soc50_Omg1e5.npy")

plt.plot(Z_0Pa_soc50_Omg1e5[:, 1], -Z_0Pa_soc50_Omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")

plt.plot(Z_1e7Pa_soc50_Omg1e5[:, 1], -Z_1e7Pa_soc50_Omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress_1e7:.0e} Pa, SOC = {initialSOC_1e7_Omg1e5:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")


plt.plot(Z_1e8Pa_soc50_Omg1e5[:, 1], -Z_1e8Pa_soc50_Omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")

# plt.plot(Z_1e9Pa_soc50_Omg1e5[:, 1], -Z_1e9Pa_soc50_Omg1e5[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa_Omg1e5 = pybamm.load("sol_0Pa_soc50_Omg1e5.pkl")
sol_1e7Pa_Omg1e5 = pybamm.load("sol_1e7Pa_soc50_Omg1e5.pkl")
sol_1e8Pa_Omg1e5 = pybamm.load("sol_1e8Pa_soc50_Omg1e5.pkl")
sol_1e9Pa_Omg1e5 = pybamm.load("sol_1e9Pa_soc50_Omg1e5.pkl")

plt.plot(sol_0Pa_Omg1e5["Time [s]"].entries, sol_0Pa_Omg1e5["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")
plt.plot(sol_1e7Pa_Omg1e5["Time [s]"].entries, sol_1e7Pa_Omg1e5["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress_1e7:.0e} Pa, SOC = {initialSOC_1e7_Omg1e5:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")
# plt.plot(sol_1e8Pa_Omg1e5["Time [s]"].entries, sol_1e8Pa_Omg1e5["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")
# plt.plot(sol_1e9Pa_Omg1e5["Time [s]"].entries, sol_1e9Pa_Omg1e5["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}, $\Omega$ = 3.1e-5 m³.mol⁻¹")   

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

## SOC = 50 %, $\Omega$ = 3.1e-4 (100x)

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0},
    check_already_exists=False  
)
param0.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6},
    check_already_exists=False  
)
param1.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7},
    check_already_exists=False  
)
param2.update(
    {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)



var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )


######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")


In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2


# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2,

         ]

y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i]
                                        )
# draw line at soc = 0.5
plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# draw line at 3.6925 V
plt.axhline(y=3.6925, color='gray', linestyle='--', label='V = 3.6925 V')
plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.title("$\Omega$ = 3.1e-4 m3.mol-1")


### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################

# rename variables
Z_0Pa_soc50_Omg1e4 = []
Z_0Pa_soc50_Omg1e4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_0Pa_soc50_Omg1e4.pkl")

# save to txt
np.save("Z_0Pa_soc50_Omg1e4", Z_0Pa_soc50_Omg1e4)
np.savetxt("Z_0Pa_soc50_Omg1e4.txt", Z_0Pa_soc50_Omg1e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = target_soc_1

#####################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
        check_already_exists=False
        )
    
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

############################
# rename variables
Z_1e6Pa_soc50_Omg1e4 = []
Z_1e6Pa_soc50_Omg1e4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e6Pa_soc50_Omg1e4.pkl")

# save to txt
np.save("Z_1e6Pa_soc50_Omg1e4", Z_1e6Pa_soc50_Omg1e4)
np.savetxt("Z_1e6Pa_soc50_Omg1e4.txt", Z_1e6Pa_soc50_Omg1e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_2

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param.update(
        {"Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################
# rename variables
Z_1e7Pa_soc50_Omg1e4 = []
Z_1e7Pa_soc50_Omg1e4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e7Pa_soc50_Omg1e4.pkl")

# save to txt
np.save("Z_1e7Pa_soc50_Omg1e4", Z_1e7Pa_soc50_Omg1e4)
np.savetxt("Z_1e7Pa_soc50_Omg1e4.txt", Z_1e7Pa_soc50_Omg1e4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### Vsi rezultati

In [ ]:
Z_0Pa_soc50_Omg1e4 = np.load("Z_0Pa_soc50_Omg1e4.npy")
Z_1e6Pa_soc50_Omg1e4 = np.load("Z_1e6Pa_soc50_Omg1e4.npy")
Z_1e7Pa_soc50_Omg1e4 = np.load("Z_1e7Pa_soc50_Omg1e4.npy")

hydrostatic_stress1 = 1e6
hydrostatic_stress2 = 1e7

initialSOC1 = target_soc_1
initialSOC2 = target_soc_2

plt.plot(Z_0Pa_soc50_Omg1e4[:, 1], -Z_0Pa_soc50_Omg1e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $\Omega$ = 3.1e-4 m³.mol⁻¹")

plt.plot(Z_1e6Pa_soc50_Omg1e4[:, 1], -Z_1e6Pa_soc50_Omg1e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}, $\Omega$ = 3.1e-4 m³.mol⁻¹")

plt.plot(Z_1e7Pa_soc50_Omg1e4[:, 1], -Z_1e7Pa_soc50_Omg1e4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}, $\Omega$ = 3.1e-4 m³.mol⁻¹")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa_Omg1e4 = pybamm.load("sol_0Pa_soc50_Omg1e4.pkl")
sol_1e6Pa_Omg1e4 = pybamm.load("sol_1e6Pa_soc50_Omg1e4.pkl")
sol_1e7Pa_Omg1e4 = pybamm.load("sol_1e7Pa_soc50_Omg1e4.pkl")

plt.plot(sol_0Pa_Omg1e4["Time [s]"].entries, sol_0Pa_Omg1e4["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $\Omega$ = 3.1e-4 m³.mol⁻¹")
plt.plot(sol_1e6Pa_Omg1e4["Time [s]"].entries, sol_1e6Pa_Omg1e4["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}, $\Omega$ = 3.1e-4 m³.mol⁻¹")
plt.plot(sol_1e7Pa_Omg1e4["Time [s]"].entries, sol_1e7Pa_Omg1e4["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}, $\Omega$ = 3.1e-4 m³.mol⁻¹")   

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

## SOC = 80 % 

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0},
    check_already_exists=False  
)


param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i]
                                        )
# draw line at soc = 0.8
plt.axvline(x=0.8, color='gray', linestyle='--', label='SOC = 0.8')

# draw line at 3.8783 V
plt.axhline(y=3.8783, color='gray', linestyle='--', label='V = 3.8783 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.8
target_soc = 0.8
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.8

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################

# rename variables
Z_0Pa_soc80 = []
Z_0Pa_soc80 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_0Pa_soc80.pkl")

# save to txt
np.save("Z_0Pa_soc80", Z_0Pa_soc80)
np.savetxt("Z_0Pa_soc80.txt", Z_0Pa_soc80)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e8 Pa

In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1

#####################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

############################
# rename variables
Z_1e8Pa_soc80 = []
Z_1e8Pa_soc80 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e8Pa_soc80.pkl")

# save to txt
np.save("Z_1e8Pa_soc80", Z_1e8Pa_soc80)
np.savetxt("Z_1e8Pa_soc80.txt", Z_1e8Pa_soc80)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e9 Pa

In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Ai2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################
# rename variables
Z_1e9Pa_soc80 = []
Z_1e9Pa_soc80 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

In [ ]:
# save results
sol_DFN.save("sol_1e9Pa_soc80.pkl")

# save to txt
np.save("Z_1e9Pa_soc80", Z_1e9Pa_soc80)
np.savetxt("Z_1e9Pa_soc80.txt", Z_1e9Pa_soc80)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### Vsi rezultati

In [ ]:
# Z_0Pa_soc80 = np.load("Z_0Pa_soc80.npy")
# Z_1e8Pa_soc80 = np.load("Z_1e8Pa_soc80.npy")
# Z_1e9Pa_soc80 = np.load("Z_1e9Pa_soc80.npy")

plt.plot(Z_0Pa_soc80[:, 1], -Z_0Pa_soc80[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc80[:, 1], -Z_1e8Pa_soc80[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e9Pa_soc80[:, 1], -Z_1e9Pa_soc80[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("sol_0Pa_soc80.pkl")
sol_1e8Pa = pybamm.load("sol_1e8Pa_soc80.pkl")
sol_1e9Pa = pybamm.load("sol_1e9Pa_soc80.pkl")

plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")
# plt.plot(sol_1e9Pa["Time [s]"].entries, sol_1e9Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")   

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

## OCV krivulja

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Ai2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Ai2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Ai2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.3
# plt.axvline(x=0.3, color='gray', linestyle='--', label='SOC = 0.3')

# draw line at 3.6391 V
# plt.axhline(y=3.6391, color='gray', linestyle='--', label='V = 3.6391 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

# NMC + SiG: EIS pri konst. napetosti

In [ ]:
# import
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

Moramo definirati $\Omega$ za katero pa ni podatkov na internetu. Trenutno je definirana enaka $\Omega$ kot pri Ai2020 za LCO+LiC6 baterijo, $\Omega$ = 3.1e-6 m3/mol.

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Chen2020") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Chen2020") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Chen2020") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Voltage [V]"].data, 
         sol_DFN_1["Voltage [V]"].data,
        sol_DFN_2["Voltage [V]"].data
         ]

plt.figure(figsize=fig_size)
for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i]
                                        )
# draw line at soc = 0.5
plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# draw line at 3.5426 V
plt.axhline(y=3.5426, color='gray', linestyle='--', label='V = 3.5426 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

## SOC = 0.5

### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Chen2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################

# rename variables
Z_0Pa_soc50 = []
Z_0Pa_soc50 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

In [ ]:
# save results
# sol_DFN.save("NMC_sol_0Pa_soc50.pkl")

# save to txt
np.save("NMC_Z_0Pa_soc50", Z_0Pa_soc50)
np.savetxt("NMC_Z_0Pa_soc50.txt", Z_0Pa_soc50)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e8 Pa

In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1
omega = 3.1e-6

#####################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Chen2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

############################
# rename variables
Z_1e8Pa_soc50 = []
Z_1e8Pa_soc50 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

In [ ]:
# save results
# sol_DFN.save("NMC_sol_1e8Pa_soc50.pkl")

# save to txt
np.save("NMC_Z_1e8Pa_soc50", Z_1e8Pa_soc50)
np.savetxt("NMC_Z_1e8Pa_soc50.txt", Z_1e8Pa_soc50)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### $\sigma_h$ = 1e9 Pa

In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Chen2020") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)

#########################################
# rename variables
Z_1e9Pa_soc50 = []
Z_1e9Pa_soc50 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

In [ ]:
# save results
# sol_DFN.save("NMC_sol_1e9Pa_soc50.pkl")

# save to txt
np.save("NMC_Z_1e9Pa_soc50", Z_1e9Pa_soc50)
np.savetxt("NMC_Z_1e9Pa_soc50.txt", Z_1e9Pa_soc50)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

## Vsi rezultati

In [ ]:
# Z_0Pa_soc50 = np.load("Z_0Pa_soc50.npy")
# Z_1e8Pa_soc50 = np.load("Z_1e8Pa_soc50.npy")
# Z_1e9Pa_soc50 = np.load("Z_1e9Pa_soc50.npy")

plt.plot(Z_0Pa_soc50[:, 1], -Z_0Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc50[:, 1], -Z_1e8Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e9Pa_soc50[:, 1], -Z_1e9Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("NMC_sol_0Pa_soc50.pkl")
sol_1e8Pa = pybamm.load("NMC_sol_1e8Pa_soc50.pkl")
sol_1e9Pa = pybamm.load("NMC_sol_1e9Pa_soc50.pkl")

plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")
plt.plot(sol_1e9Pa["Time [s]"].entries, sol_1e9Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")   

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

# NMC + LiC6: EIS pri konst. napetosti

In [ ]:
# import
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize
from joblib import Parallel, delayed
from scipy.signal import savgol_filter

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

## SOC = 0.5

### $\Omega$ = 3.1e-6 $m^3/mol$

Moramo definirati $\Omega$ za katero pa ni podatkov na internetu. Trenutno je definirana enaka $\Omega$ kot pri Ai2020 za LCO+LiC6 baterijo, $\Omega$ = 3.1e-6 m3/mol.

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.5
# plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# # draw line at 3.7156 V
# plt.axhline(y=3.7156, color='gray', linestyle='--', label='V = 3.7156 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izrišemo še dQ/dV

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],

         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]



plt.figure(figsize=(9, 8))
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e6 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e7 Pa", 

                                                                                ][i], 
                                        linestyle=["-", "-", "-", ][i], color=color)

    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
plt.grid()
plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))


#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
# hydrostatic_stress = 0 # Pa
# initialSOC = 0.5
# omega = 3.1e-6

# ######################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc50_OCV = []
Z_0Pa_soc50_OCV = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

# # save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc50_OCV.pkl")

# # save to txt
# np.save("NMC_LiC6_Z_0Pa_soc50_OCV", Z_0Pa_soc50_OCV)
# np.savetxt("NMC_LiC6_Z_0Pa_soc50_OCV.txt", Z_0Pa_soc50_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e8 Pa

In [ ]:
# hydrostatic_stress = 1e8 # Pa
# initialSOC = target_soc_1
# omega = 3.1e-6

# #####################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1
omega = 3.1e-6

######################################
I_0 = 0.5 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 50 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 1000


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e8Pa_soc50_OCV = []
Z_1e8Pa_soc50_OCV = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e8Pa_soc50_OCV_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e8Pa_soc50_OCV_lowF", Z_1e8Pa_soc50_OCV)
np.savetxt("NMC_LiC6_Z_1e8Pa_soc50_OCV_lowF.txt", Z_1e8Pa_soc50_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e9 Pa

In [ ]:
# hydrostatic_stress = 1e9 # Pa
# initialSOC = target_soc_2
# omega = 3.1e-6

# ######################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 200 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2
omega = 3.1e-6

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.001
high_frequency = 1000
Number_of_Nyquist_points = 100

In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e9Pa_soc50_OCV = []
Z_1e9Pa_soc50_OCV = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e9Pa_soc50_OCV_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e9Pa_soc50_OCV_lowF", Z_1e9Pa_soc50_OCV)
np.savetxt("NMC_LiC6_Z_1e9Pa_soc50_OCV_lowF.txt", Z_1e9Pa_soc50_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
# Z_0Pa_soc50_OCV = np.load("NMC_LiC6_Z_0Pa_soc50_OCV.npy")
# Z_1e8Pa_soc50_OCV = np.load("NMC_LiC6_Z_1e8Pa_soc50_OCV.npy")
# Z_1e6Pa_soc50_OCV = np.load("NMC_LiC6_Z_1e6Pa_soc50_OCV.npy")
# hydrostatic_stress0 = 0
# initialSOC0 = 0.5
# hydrostatic_stress1 = 1e8
# initialSOC1 = target_soc_1
# hydrostatic_stress2 = 1e9
# initialSOC2 = target_soc_2

plt.plot(Z_0Pa_soc50_OCV[:, 1], -Z_0Pa_soc50_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc50_OCV[:, 1], -Z_1e8Pa_soc50_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e9Pa_soc50_OCV[:, 1], -Z_1e9Pa_soc50_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.grid()
plt.show()

Preverimo, če so enake napetosti

In [ ]:
# sol_0Pa = pybamm.load("NMC_LiC6_sol_0Pa_soc50_OCV.pkl")
# sol_1e8Pa = pybamm.load("NMC_LiC6_sol_1e8Pa_soc50_OCV.pkl")
sol_1e9Pa = pybamm.load("NMC_LiC6_sol_1e9Pa_soc50_OCV.pkl")
sol_1e9Pa2 = pybamm.load("NMC_LiC6_sol_1e9Pa_soc50_OCV2.pkl")
sol_1e9Pa3 = pybamm.load("NMC_LiC6_sol_1e9Pa_soc50_OCV3.pkl")
sol_1e9Pa4 = pybamm.load("NMC_LiC6_sol_1e9Pa_soc50_OCV4.pkl")


# plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
# plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")
plt.plot(sol_1e9Pa["Time [s]"].entries, sol_1e9Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")   
plt.plot(sol_1e9Pa2["Time [s]"].entries, sol_1e9Pa2["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")   
plt.plot(sol_1e9Pa3["Time [s]"].entries, sol_1e9Pa3["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")
plt.plot(sol_1e9Pa4["Time [s]"].entries, sol_1e9Pa4["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

### $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                        "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4,
     },
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": -1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": -1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
experiment_C = pybamm.Experiment(["Charge at C/50 until 4.2 V"])

solver = pybamm.IDAKLUSolver()  


# discharge battery
sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

# charge battery
sim_DFN_0_C = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, experiment=experiment_C, solver=solver)
sim_DFN_1_C = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, experiment=experiment_C, solver=solver)
sim_DFN_2_C = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, experiment=experiment_C, solver=solver)

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_C = sim_DFN_0_C.solve(initial_soc=0)  
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1_C = sim_DFN_1_C.solve(initial_soc=0)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2_C = sim_DFN_2_C.solve(initial_soc=0)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

Q_discharged = sol_DFN_C["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_C = (Q_discharged / Q_max)

Q_discharged = sol_DFN_1_C["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1_C = (Q_discharged / Q_max)

Q_discharged = sol_DFN_2_C["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2_C = (Q_discharged / Q_max)

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1e6 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1e7 Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# # draw line at soc = 0.5
# plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# # draw line at 3.7156 V
# plt.axhline(y=3.7156, color='gray', linestyle='--', label='V = 3.7156 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
# plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.legend()
# plt.grid()

# save to eps
plt.savefig("NMC_LiC6_OCV.eps", format='eps',dpi= 300, bbox_inches='tight')

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)

dQ_dV_C = 1/savgol_filter(np.gradient(sol_DFN_C["Voltage [V]"].entries [102:], DOD_C[102:]),10,2)
dQ_dV_1_C = 1/savgol_filter(np.gradient(sol_DFN_1_C["Voltage [V]"].entries[111:],DOD_1_C [111:]),10,2)
dQ_dV_2_C = 1/savgol_filter(np.gradient(sol_DFN_2_C["Voltage [V]"].entries[122:],DOD_2_C [122:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],
         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]

x_val_C = [sol_DFN_C["Voltage [V]"].entries[102:],
        sol_DFN_1_C["Voltage [V]"].entries[111:],
        sol_DFN_2_C["Voltage [V]"].entries[122:],
         ]

y_val_C = [dQ_dV_C,
        dQ_dV_1_C,
        dQ_dV_2_C,
         ]


plt.figure(figsize=fig_size)
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1 MPa", 
                                        "$\sigma_{\mathrm{h}}=$10 MPa", 

                                                                                ][i], 
                                        linestyle=["-", "--", "-", ][i], color=color)
for i in range(len(x_val_C)):
    color = cmap(i % 6)
    plt.plot(x_val_C[i], -y_val_C[i],linestyle=["-", "--", "-", "-", "-"][i],  color=color)
    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
# plt.grid()
# plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))
plt.legend()

# save to eps
plt.savefig("NMC_LiC6_dQdV.eps", format='eps',dpi= 300, bbox_inches='tight')


In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc50_omgE4 = []
Z_0Pa_soc50_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save to txt
# np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici", Z_0Pa_soc50_omgE4)
# np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici.txt", Z_0Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = -1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc50_omgE4 = []
Z_1e6Pa_soc50_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_neg1e6Pa_soc50_omgE4_Clerici", Z_1e6Pa_soc50_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_neg1e6Pa_soc50_omgE4_Clerici.txt", Z_1e6Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = -1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc50_omgE4 = []
Z_1e7Pa_soc50_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_neg1e7Pa_soc50_omgE4_Clerici", Z_1e7Pa_soc50_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_neg1e7Pa_soc50_omgE4_Clerici.txt", Z_1e7Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

## Vsi rezultati

In [ ]:
# Z_0Pa_soc50_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_0Pa_soc50_omgE4_lowF.npy") 
# Z_1e6Pa_soc50_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF.npy")
# Z_1e7Pa_soc50_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_1e7Pa_soc50_omgE4_lowF.npy")

# Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_lowF_Clerici.npy") 
# Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF_Clerici.npy")
# Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_lowF_Clerici.npy")

# # tensile stress
# Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici.npy") 
# Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici.npy")
# Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici.npy")

# compressive stress
Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici.npy") 
Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_neg1e6Pa_soc50_omgE4_Clerici.npy")
Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_neg1e7Pa_soc50_omgE4_Clerici.npy")



# plt.plot(Z_0Pa_soc50_omgE4[:, 1], -Z_0Pa_soc50_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

# plt.plot(Z_1e6Pa_soc50_omgE4[:, 1], -Z_1e6Pa_soc50_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

# plt.plot(Z_1e7Pa_soc50_omgE4[:, 1], -Z_1e7Pa_soc50_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")

plt.plot(Z_0Pa_soc50_omgE4_Clerici[:, 1], -Z_0Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa Clerici")

plt.plot(Z_1e6Pa_soc50_omgE4_Clerici[:, 1], -Z_1e6Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa Clerici")

plt.plot(Z_1e7Pa_soc50_omgE4_Clerici[:, 1], -Z_1e7Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa Clerici")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.72 V')
plt.xlim(0, 0.012)
plt.ylim(0, 0.012)
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc50_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


## SOC = 0.2

### $\Omega$ = 3.1e-6 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.2
plt.axvline(x=0.2, color='gray', linestyle='--', label='SOC = 0.2')

# draw line at 3.5862 V
plt.axhline(y=3.5862, color='gray', linestyle='--', label='V = 3.5862 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

#### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 20%.

In [ ]:
# find soc value closest to 0.2
target_soc = 0.2
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.2
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=-1)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc20_OCV = []
Z_0Pa_soc20_OCV = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_0Pa_soc20_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc20_OCV", Z_0Pa_soc20_OCV)
np.savetxt("NMC_LiC6_Z_0Pa_soc20_OCV.txt", Z_0Pa_soc20_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e8 Pa

In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1
omega = 3.1e-6

#####################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)


In [ ]:
# rename variables
Z_1e8Pa_soc20_OCV = []
Z_1e8Pa_soc20_OCV = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e8Pa_soc20_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e8Pa_soc20_OCV", Z_1e8Pa_soc20_OCV)
np.savetxt("NMC_LiC6_Z_1e8Pa_soc20_OCV.txt", Z_1e8Pa_soc20_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
Z_0Pa_soc20 = np.load("NMC_LiC6_Z_0Pa_soc20.npy")
Z_1e8Pa_soc20_OCV = np.load("NMC_LiC6_Z_1e8Pa_soc20.npy")


plt.plot(Z_0Pa_soc20[:, 1], -Z_0Pa_soc20[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc20_OCV[:, 1], -Z_1e8Pa_soc20_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("NMC_LiC6_sol_0Pa_soc20.pkl")
sol_1e8Pa = pybamm.load("NMC_LiC6_sol_1e8Pa_soc20_OCV.pkl")


plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

## SOC = 0.25

### $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.25
plt.axvline(x=0.25, color='gray', linestyle='--', label='SOC = 0.25')

# draw line at 3.6211 V
plt.axhline(y=3.6211, color='gray', linestyle='--', label='V = 3.6211 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 25%.

In [ ]:
# find soc value closest to 0.25
target_soc = 0.25
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
# hydrostatic_stress = 0 # Pa
# initialSOC = 0.25
# omega = 3.1e-4

# ######################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.25
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc25_omgE4 = []
Z_0Pa_soc25_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_0Pa_soc25_omgE4.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc25_omgE4", Z_0Pa_soc25_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc25_omgE4.txt", Z_0Pa_soc25_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
# hydrostatic_stress = 1e6 # Pa
# initialSOC = target_soc_1
# omega = 3.1e-4

# #####################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc25_omgE4 = []
Z_1e6Pa_soc25_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc25_omgE4.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc25_omgE4", Z_1e6Pa_soc25_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc25_omgE4.txt", Z_1e6Pa_soc25_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
# hydrostatic_stress = 1e7 # Pa
# initialSOC = target_soc_2
# omega = 3.1e-4

# ######################################
# I_0 = 0.9 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc25_omgE4 = []
Z_1e7Pa_soc25_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_1e7Pa_soc25_omgE4.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e7Pa_soc25_omgE4", Z_1e7Pa_soc25_omgE4)
np.savetxt("NMC_LiC6_Z_1e7Pa_soc25_omgE4.txt", Z_1e7Pa_soc25_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
# Z_0Pa_soc25_omgE4 = np.load("NMC_LiC6_Z_0Pa_soc25_omgE4_lowF.npy") 
# Z_1e6Pa_soc25_omgE4 = np.load("NMC_LiC6_Z_1e6Pa_soc25_omgE4_lowF.npy")
# Z_1e7Pa_soc25_omgE4 = np.load("NMC_LiC6_Z_1e7Pa_soc25_omgE4_lowF.npy")
# # hydrostatic_stress0 = 0
# initialSOC0 = 0.25
# hydrostatic_stress1 = 1e6
# initialSOC1 = target_soc_1
# hydrostatic_stress2 = 1e7
# initialSOC2 = target_soc_2

plt.plot(Z_0Pa_soc25_omgE4[:, 1], -Z_0Pa_soc25_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa")

plt.plot(Z_1e6Pa_soc25_omgE4[:, 1], -Z_1e6Pa_soc25_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa")

plt.plot(Z_1e7Pa_soc25_omgE4[:, 1], -Z_1e7Pa_soc25_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
# plt.legend(loc='center right', bbox_to_anchor=(1.7, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.xlim(0, 0.025)
plt.ylim(0, 0.025)
plt.title('Nyquist Plot @ 3.62 V')
plt.grid()
plt.show()

## SOC = 0.3

### $\Omega$ = 3.1e-6 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.3
plt.axvline(x=0.3, color='gray', linestyle='--', label='SOC = 0.3')

# draw line at 3.6391 V
plt.axhline(y=3.6391, color='gray', linestyle='--', label='V = 3.6391 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Izrišemo še dQ/dV

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],

         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]



plt.figure(figsize=(9, 8))
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e6 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e7 Pa", 

                                                                                ][i], 
                                        linestyle=["-", "-", "-", ][i], color=color)

    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
plt.grid()
plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))


#### $\sigma_h$ = 0 Pa

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 30%.

In [ ]:
# find soc value closest to 0.3
target_soc = 0.3
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.3
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc30_OCV2 = []
Z_0Pa_soc30_OCV2 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc30_OCV.pkl")

# # save to txt
# np.save("NMC_LiC6_Z_0Pa_soc30_OCV", Z_0Pa_soc30_OCV)
# np.savetxt("NMC_LiC6_Z_0Pa_soc30_OCV.txt", Z_0Pa_soc30_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e8 Pa

In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1
omega = 3.1e-6

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 50 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=-1)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e8Pa_soc30_OCV = []
Z_1e8Pa_soc30_OCV = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e8Pa_soc30_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e8Pa_soc30_OCV", Z_1e8Pa_soc30_OCV)
np.savetxt("NMC_LiC6_Z_1e8Pa_soc30_OCV.txt", Z_1e8Pa_soc30_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e9 Pa

In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2
omega = 3.1e-6

######################################
I_0 = 0.7 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 70 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e9Pa_soc30_OCV = []
Z_1e9Pa_soc30_OCV = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e9Pa_soc30_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e9Pa_soc30_OCV", Z_1e9Pa_soc30_OCV)
np.savetxt("NMC_LiC6_Z_1e9Pa_soc30_OCV.txt", Z_1e9Pa_soc30_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
Z_0Pa_soc30_OCV = np.load("NMC_LiC6_Z_0Pa_soc30_OCV.npy")
# Z_1e8Pa_soc30_OCV = np.load("NMC_LiC6_Z_1e8Pa_soc30_OCV.npy")
# hydrostatic_stress0 = 0
# initialSOC0 = 0.3
# hydrostatic_stress1 = 1e8
# initialSOC1 = target_soc_1
# hydrostatic_stress2 = 1e9
# initialSOC2 = target_soc_2


plt.plot(Z_0Pa_soc30_OCV[:, 1], -Z_0Pa_soc30_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc30_OCV[:, 1], -Z_1e8Pa_soc30_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e9Pa_soc30_OCV[:, 1], -Z_1e9Pa_soc30_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.grid()
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("NMC_LiC6_sol_0Pa_soc30_OCV.pkl")
sol_1e8Pa = pybamm.load("NMC_LiC6_sol_1e8Pa_soc30_OCV.pkl")

plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
# plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

### $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.3
plt.axvline(x=0.3, color='gray', linestyle='--', label='SOC = 0.3')

# draw line at 3.6391 V
plt.axhline(y=3.6391, color='gray', linestyle='--', label='V = 3.6391 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 30%.

In [ ]:
# find soc value closest to 0.3
target_soc = 0.3
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.3
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc30_omgE4 = []
Z_0Pa_soc30_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici", Z_0Pa_soc30_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici.txt", Z_0Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc30_omgE4 = []
Z_1e6Pa_soc30_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici", Z_1e6Pa_soc30_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici.txt", Z_1e6Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc30_omgE4 = []
Z_1e7Pa_soc30_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici", Z_1e7Pa_soc30_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici.txt", Z_1e7Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

## Vsi rezultati

In [ ]:
# Z_0Pa_soc30_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_0Pa_soc30_omgE4_lowF.npy") 
# Z_1e6Pa_soc30_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF.npy")
# Z_1e7Pa_soc30_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_1e7Pa_soc30_omgE4_lowF.npy")

# Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_lowF_Clerici.npy") 
# Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF_Clerici.npy")
# Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_lowF_Clerici.npy")

Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici.npy") 
Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici.npy")
Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici.npy")

# plt.plot(Z_0Pa_soc30_omgE4[:, 1], -Z_0Pa_soc30_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

# plt.plot(Z_1e6Pa_soc30_omgE4[:, 1], -Z_1e6Pa_soc30_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

# plt.plot(Z_1e7Pa_soc30_omgE4[:, 1], -Z_1e7Pa_soc30_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")

plt.plot(Z_0Pa_soc30_omgE4_Clerici[:, 1], -Z_0Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa Clerici")

plt.plot(Z_1e6Pa_soc30_omgE4_Clerici[:, 1], -Z_1e6Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa Clerici")

plt.plot(Z_1e7Pa_soc30_omgE4_Clerici[:, 1], -Z_1e7Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa Clerici")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.64 V')
plt.xlim(0, 0.012)
plt.ylim(0, 0.012)
# plt.grid()

# # save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc30_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


## SOC = 0.7

### $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.7
# plt.axvline(x=0.7, color='gray', linestyle='--', label='SOC = 0.7')

# draw line at 3.8788 V
# plt.axhline(y=3.8788, color='gray', linestyle='--', label='V = 3.8788 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],

         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]



plt.figure(figsize=(9, 8))
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e6 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e7 Pa", 

                                                                                ][i], 
                                        linestyle=["-", "-", "-", ][i], color=color)

    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
plt.grid()
plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))


Pogledamo pri kateri napetosti neobremenjene celice je SOC = 80%.

In [ ]:
# find soc value closest to 0.7
target_soc = 0.7
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()-1 # korekcija, da mi ne da manjšega SOCa
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()-1 # korekcija, da mi ne da manjšega SOCa
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
# hydrostatic_stress = 0 # Pa
# initialSOC = 0.7
# omega = 3.1e-4

# ######################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.7
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc70_omgE4 = []
Z_0Pa_soc70_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_0Pa_soc70_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc70_omgE4_lowF", Z_0Pa_soc70_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc70_omgE4_lowF.txt", Z_0Pa_soc70_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
# hydrostatic_stress = 1e6 # Pa
# initialSOC = target_soc_1
# omega = 3.1e-4

# #####################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc70_omgE4 = []
Z_1e6Pa_soc70_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc70_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc70_omgE4_lowF", Z_1e6Pa_soc70_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc70_omgE4_lowF.txt", Z_1e6Pa_soc70_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc70_omgE4 = []
Z_1e7Pa_soc70_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_1e7Pa_soc70_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e7Pa_soc70_omgE4_lowF", Z_1e7Pa_soc70_omgE4)
np.savetxt("NMC_LiC6_Z_1e7Pa_soc70_omgE4_lowF.txt", Z_1e7Pa_soc70_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
# Z_0Pa_soc70_omgE4 = np.load("NMC_LiC6_Z_0Pa_soc70_omgE4_lowF.npy")
# Z_1e6Pa_soc70_omgE4 = np.load("NMC_LiC6_Z_1e6Pa_soc70_omgE4_lowF.npy")
# Z_1e7Pa_soc70_omgE4 = np.load("NMC_LiC6_Z_1e7Pa_soc70_omgE4_lowF.npy")
# hydrostatic_stress0 = 0
# initialSOC0 = 0.8
# hydrostatic_stress1 = 1e6
# initialSOC1 = target_soc_1
# hydrostatic_stress2 = 1e7
# initialSOC2 = target_soc_2

plt.plot(Z_0Pa_soc70_omgE4[:, 1], -Z_0Pa_soc70_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e6Pa_soc70_omgE4[:, 1], -Z_1e6Pa_soc70_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e7Pa_soc70_omgE4[:, 1], -Z_1e7Pa_soc70_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('Nyquist Plot @ 3.88 V')
plt.xlim(0, 0.085)
plt.ylim(0, 0.085)
plt.grid()
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("NMC_LiC6_sol_0Pa_soc80.pkl")
sol_1e7Pa = pybamm.load("NMC_LiC6_sol_1e7Pa_soc80_OCV.pkl")


plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
plt.plot(sol_1e7Pa["Time [s]"].entries, sol_1e7Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

## SOC = 0.75

### $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.75
plt.axvline(x=0.75, color='gray', linestyle='--', label='SOC = 0.75')

# draw line at 3.9284 V
plt.axhline(y=3.9284, color='gray', linestyle='--', label='V = 3.9284 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 75 %.

In [ ]:
# find soc value closest to 0.75
target_soc = 0.75
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
# hydrostatic_stress = 0 # Pa
# initialSOC = 0.75
# omega = 3.1e-4

# ######################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.75
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc75_omgE4 = []
Z_0Pa_soc75_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_0Pa_soc75_omgE4.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc75_omgE4", Z_0Pa_soc75_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc75_omgE4.txt", Z_0Pa_soc75_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
# hydrostatic_stress = 1e6 # Pa
# initialSOC = target_soc_1
# omega = 3.1e-4

# #####################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc75_omgE4 = []
Z_1e6Pa_soc75_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc75_omgE4.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc75_omgE4", Z_1e6Pa_soc75_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc75_omgE4.txt", Z_1e6Pa_soc75_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
# hydrostatic_stress = 1e7 # Pa
# initialSOC = target_soc_2
# omega = 3.1e-4

# ######################################
# I_0 = 0.9 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc75_omgE4 = []
Z_1e7Pa_soc75_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save results
# sol_DFN.save("NMC_LiC6_sol_1e7Pa_soc75_omgE4.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e7Pa_soc75_omgE4", Z_1e7Pa_soc75_omgE4)
np.savetxt("NMC_LiC6_Z_1e7Pa_soc75_omgE4.txt", Z_1e7Pa_soc75_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
# Z_0Pa_soc75_omgE4 = np.load("NMC_LiC6_Z_0Pa_soc75_omgE4_lowF.npy") 
# Z_1e6Pa_soc75_omgE4 = np.load("NMC_LiC6_Z_1e6Pa_soc75_omgE4_lowF.npy")
# Z_1e7Pa_soc75_omgE4 = np.load("NMC_LiC6_Z_1e7Pa_soc75_omgE4_lowF.npy")
# hydrostatic_stress0 = 0
# initialSOC0 = 0.75
# hydrostatic_stress1 = 1e6
# initialSOC1 = target_soc_1
# hydrostatic_stress2 = 1e7
# initialSOC2 = target_soc_2

plt.plot(Z_0Pa_soc75_omgE4[:, 1], -Z_0Pa_soc75_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa")

plt.plot(Z_1e6Pa_soc75_omgE4[:, 1], -Z_1e6Pa_soc75_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa")

plt.plot(Z_1e7Pa_soc75_omgE4[:, 1], -Z_1e7Pa_soc75_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
# plt.legend(loc='center right', bbox_to_anchor=(1.7, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.xlim(0, 0.025)
plt.ylim(0, 0.025)
plt.title('Nyquist Plot @ 3.93 V')
plt.grid()
plt.show()

## SOC = 0.8

### $\Omega$ = 3.1e-6 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e8,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e9,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.8
plt.axvline(x=0.8, color='gray', linestyle='--', label='SOC = 0.8')

# draw line at 3.9803 V
plt.axhline(y=3.9803, color='gray', linestyle='--', label='V = 3.9803 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Izrišemo še dQ/dV

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],

         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]



plt.figure(figsize=(9, 8))
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e6 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e7 Pa", 

                                                                                ][i], 
                                        linestyle=["-", "-", "-", ][i], color=color)

    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
plt.grid()
plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))


Pogledamo pri kateri napetosti neobremenjene celice je SOC = 80%.

In [ ]:
# find soc value closest to 0.8
target_soc = 0.8
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
# hydrostatic_stress = 0 # Pa
# initialSOC = 0.8
# omega = 3.1e-5

# ######################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.8
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc80_OCV = []
Z_0Pa_soc80_OCV = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_0Pa_soc80_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc80_OCV", Z_0Pa_soc80_OCV)
np.savetxt("NMC_LiC6_Z_0Pa_soc80_OCV.txt", Z_0Pa_soc80_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e8 Pa

In [ ]:
# hydrostatic_stress = 1e8 # Pa
# initialSOC = target_soc_1
# omega = 3.1e-6

# #####################################
# I_0 = 0.1 #sampling current amplitude
# N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
# Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.1
# high_frequency = 1000
# Number_of_Nyquist_points = 100

# #initialisation of table where impedances will be stored
# Z = []

# #freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
# freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

# #for loop acros all measured frequencies
# for i in range(0,len(freq_points)):

#     freq = freq_points[i]
    
#     t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
#     #Generation of the current signal used for sampling
#     sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
#     # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
#     # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
#     def my_fun(I0, freq):
#         def current(t):
#             return I0 * pybamm.sin(2 * np.pi * freq * t)
#         return current
    

#     #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


#     model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
#                                                 "particle": "Fickian diffusion",
#                                                 },
#                                         )

#     param = pybamm.ParameterValues("Mohtat2020_Mech") 
#     param.update(
#         {"Hydrostatic stress [Pa]": hydrostatic_stress,
#          "Negative electrode partial molar volume [m3.mol-1]": omega},
#         check_already_exists=False  
#     )
#     param["Current function [A]"] = my_fun(I_0, freq)

#     var_pts = {
#     "x_n": 30,  # negative electrode
#     "x_s": 20,  # separator
#     "x_p": 30,  # positive electrode
#     "r_n": 25,  # negative particle
#     "r_p": 25,  # positive particle
# }

#     # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
#     solver = pybamm.IDAKLUSolver()  

#     sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
#                                 )

#     sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

#     simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


#     #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
#     #Preparing guess that will be used as a starting point for the fitting function
#     guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
#     guess_phase = 0 #gues for the phase shift is set to 0
#     guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


#     # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
#     optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
#     #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
#     est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
#     #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
#     #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
#     Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

# Z = np.array(Z)


In [ ]:
hydrostatic_stress = 1e8 # Pa
initialSOC = target_soc_1
omega = 3.1e-6

######################################
I_0 = 0.5 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 50 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e8Pa_soc80_OCV = []
Z_1e8Pa_soc80_OCV = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e8Pa_soc80_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e8Pa_soc80_OCV", Z_1e8Pa_soc80_OCV)
np.savetxt("NMC_LiC6_Z_1e8Pa_soc80_OCV.txt", Z_1e8Pa_soc80_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e9 Pa

In [ ]:
hydrostatic_stress = 1e9 # Pa
initialSOC = target_soc_2
omega = 3.1e-6

######################################
I_0 = 1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e9Pa_soc80_OCV = []
Z_1e9Pa_soc80_OCV = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_1e9Pa_soc80_OCV.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e9Pa_soc80_OCV", Z_1e9Pa_soc80_OCV)
np.savetxt("NMC_LiC6_Z_1e9Pa_soc80_OCV.txt", Z_1e9Pa_soc80_OCV)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### Vsi rezultati

In [ ]:
# Z_0Pa_soc80_OCV = np.load("NMC_LiC6_Z_0Pa_soc80_OCV.npy")
# Z_1e8Pa_soc80_OCV = np.load("NMC_LiC6_Z_1e8Pa_soc80_OCV.npy")
# Z_1e9Pa_soc80_OCV = np.load("NMC_LiC6_Z_1e9Pa_soc80_OCV.npy")
# hydrostatic_stress0 = 0
# initialSOC0 = 0.8
# hydrostatic_stress1 = 1e8
# initialSOC1 = target_soc_1
# hydrostatic_stress2 = 1e9
# initialSOC2 = target_soc_2

plt.plot(Z_0Pa_soc80_OCV[:, 1], -Z_0Pa_soc80_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")

plt.plot(Z_1e8Pa_soc80_OCV[:, 1], -Z_1e8Pa_soc80_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.plot(Z_1e9Pa_soc80_OCV[:, 1], -Z_1e9Pa_soc80_OCV[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress2:.0e} Pa, SOC = {initialSOC2:.2f}")

# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.grid()
plt.show()

Preverimo, če so enake napetosti

In [ ]:
sol_0Pa = pybamm.load("NMC_LiC6_sol_0Pa_soc80.pkl")
sol_1e8Pa = pybamm.load("NMC_LiC6_sol_1e8Pa_soc80_OCV.pkl")


plt.plot(sol_0Pa["Time [s]"].entries, sol_0Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}")
plt.plot(sol_1e8Pa["Time [s]"].entries, sol_1e8Pa["Voltage [V]"].entries, label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress1:.0e} Pa, SOC = {initialSOC1:.2f}")

plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

### $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.8
plt.axvline(x=0.8, color='gray', linestyle='--', label='SOC = 0.8')

# draw line at 3.9803 V
plt.axhline(y=3.9803, color='gray', linestyle='--', label='V = 3.9803 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

In [ ]:
dQ_dV = 1/savgol_filter(np.gradient(sol_DFN["Voltage [V]"].entries [70:], DOD[70:]),10,2)
dQ_dV_1 = 1/savgol_filter(np.gradient(sol_DFN_1["Voltage [V]"].entries[75:],DOD_1 [75:]),10,2)
dQ_dV_2 = 1/savgol_filter(np.gradient(sol_DFN_2["Voltage [V]"].entries[80:],DOD_2 [80:]),10,2)


# plot
x_val = [sol_DFN["Voltage [V]"].entries[70:],
        sol_DFN_1["Voltage [V]"].entries[75:],
        sol_DFN_2["Voltage [V]"].entries[80:],

         ]

y_val = [dQ_dV,
        dQ_dV_1,
        dQ_dV_2,
         ]



plt.figure(figsize=(9, 8))
cmap = plt.get_cmap('tab10')
for i in range(len(x_val)):
    color = cmap(i % 10)
    plt.plot(x_val[i], -y_val[i], label=["$\sigma_{\mathrm{h}}=$0 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e6 Pa", 
                                        "$\sigma_{\mathrm{h}}=$1e7 Pa", 

                                                                                ][i], 
                                        linestyle=["-", "-", "-", ][i], color=color)

    
# plt.xlim(4.15, 4.154)
# plt.ylim(-1e6, 2e6)
plt.xlabel("V [V]")
plt.ylabel("dQ/dV [As/V]")
plt.grid()
plt.legend(loc="center right", bbox_to_anchor=(1.4, 0.5))


Pogledamo pri kateri napetosti neobremenjene celice je SOC = 80%.

In [ ]:
# find soc value closest to 0.8
target_soc = 0.8
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.8
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc80_omgE4 = []
Z_0Pa_soc80_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici", Z_0Pa_soc80_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici.txt", Z_0Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc80_omgE4 = []
Z_1e6Pa_soc80_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici", Z_1e6Pa_soc80_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici.txt", Z_1e6Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = 1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc80_omgE4 = []
Z_1e7Pa_soc80_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici", Z_1e7Pa_soc80_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici.txt", Z_1e7Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

## Vsi rezultati

In [ ]:
# Z_0Pa_soc80_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_0Pa_soc80_omgE4_lowF.npy") 
# Z_1e6Pa_soc80_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF.npy")
# Z_1e7Pa_soc80_omgE4 = np.load("NMC_LiC6 results/withoutClerici/NMC_LiC6_Z_1e7Pa_soc80_omgE4_lowF.npy")

# Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_lowF_Clerici.npy") 
# Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF_Clerici.npy")
# Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_lowF_Clerici.npy")

Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici.npy") 
Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici.npy")
Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici.npy")

# plt.plot(Z_0Pa_soc80_omgE4[:, 1], -Z_0Pa_soc80_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

# plt.plot(Z_1e6Pa_soc80_omgE4[:, 1], -Z_1e6Pa_soc80_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

# plt.plot(Z_1e7Pa_soc80_omgE4[:, 1], -Z_1e7Pa_soc80_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
#         label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")

plt.plot(Z_0Pa_soc80_omgE4_Clerici[:, 1], -Z_0Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa Clerici")

plt.plot(Z_1e6Pa_soc80_omgE4_Clerici[:, 1], -Z_1e6Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa Clerici")

plt.plot(Z_1e7Pa_soc80_omgE4_Clerici[:, 1], -Z_1e7Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa Clerici")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.98 V')
plt.xlim(0, 0.012)
plt.ylim(0, 0.012)
# plt.grid()

# # save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc80_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


## $\sigma_h$ = 0e9 Pa

#### SOC = 0.99

Računam pri SOC 99% ker pri SOC 100% da error: Events ['Maximum voltage [V]'] are non-positive at initial conditions.

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.99
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc99_omgE4 = []
Z_0Pa_soc99_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc99_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc99_omgE4_lowF", Z_0Pa_soc99_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc99_omgE4_lowF.txt", Z_0Pa_soc99_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.9

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.9
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc90_omgE4 = []
Z_0Pa_soc90_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc90_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc90_omgE4_lowF", Z_0Pa_soc90_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc90_omgE4_lowF.txt", Z_0Pa_soc90_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.8

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.8
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc80_omgE4 = []
Z_0Pa_soc80_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc80_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc80_omgE4_lowF", Z_0Pa_soc80_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc80_omgE4_lowF.txt", Z_0Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.7

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.7
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc70_omgE4 = []
Z_0Pa_soc70_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc70_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc70_omgE4_lowF", Z_0Pa_soc70_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc70_omgE4_lowF.txt", Z_0Pa_soc70_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.6

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.6
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc60_omgE4 = []
Z_0Pa_soc60_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc60_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc60_omgE4_lowF", Z_0Pa_soc60_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc60_omgE4_lowF.txt", Z_0Pa_soc60_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.5

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc50_omgE4 = []
Z_0Pa_soc50_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc50_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc50_omgE4_lowF", Z_0Pa_soc50_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc50_omgE4_lowF.txt", Z_0Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.4

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.4
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc40_omgE4 = []
Z_0Pa_soc40_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc40_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc40_omgE4_lowF", Z_0Pa_soc40_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc40_omgE4_lowF.txt", Z_0Pa_soc40_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.3

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.3
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc30_omgE4 = []
Z_0Pa_soc30_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc30_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc30_omgE4_lowF", Z_0Pa_soc30_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc30_omgE4_lowF.txt", Z_0Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.2

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.2
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc20_omgE4 = []
Z_0Pa_soc20_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc20_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc20_omgE4_lowF", Z_0Pa_soc20_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc20_omgE4_lowF.txt", Z_0Pa_soc20_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.1

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.1
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc10_omgE4 = []
Z_0Pa_soc10_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc10_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc10_omgE4_lowF", Z_0Pa_soc10_omgE4)
np.savetxt("NMC_LiC6_Z_0Pa_soc10_omgE4_lowF.txt", Z_0Pa_soc10_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### Vsi rezultati

In [ ]:
# plt.plot(Z_0Pa_soc99_omgE4[:, 1], -Z_0Pa_soc99_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.99")
# plt.plot(Z_0Pa_soc90_omgE4[:, 1], -Z_0Pa_soc90_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.9")
# plt.plot(Z_0Pa_soc80_omgE4[:, 1], -Z_0Pa_soc80_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.8")
# plt.plot(Z_0Pa_soc70_omgE4[:, 1], -Z_0Pa_soc70_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.7")
# plt.plot(Z_0Pa_soc60_omgE4[:, 1], -Z_0Pa_soc60_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.6")
plt.plot(Z_0Pa_soc50_omgE4[:, 1], -Z_0Pa_soc50_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.5")
plt.plot(Z_0Pa_soc40_omgE4[:, 1], -Z_0Pa_soc40_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.4")
plt.plot(Z_0Pa_soc30_omgE4[:, 1], -Z_0Pa_soc30_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.3")
plt.plot(Z_0Pa_soc20_omgE4[:, 1], -Z_0Pa_soc20_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.2")
plt.plot(Z_0Pa_soc10_omgE4[:, 1], -Z_0Pa_soc10_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.1")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('NMC+LiC6 ($\sigma_{\mathrm{h}}=0$ Pa)')
# plt.xlim(0.01, 0.025)
# plt.ylim(0.01, 0.025)
plt.xlim(0, 0.06)
plt.ylim(0, 0.06)

plt.legend(loc = "center right", bbox_to_anchor=(1.6, 0.5))

## $\sigma_h$ = 1e6 Pa

#### SOC = 1

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.99
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc99_omgE4 = []
Z_1e6Pa_soc99_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc99_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc99_omgE4_lowF", Z_1e6Pa_soc99_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc99_omgE4_lowF.txt", Z_1e6Pa_soc99_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.9

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.9
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc90_omgE4 = []
Z_1e6Pa_soc90_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc90_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc90_omgE4_lowF", Z_1e6Pa_soc90_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc90_omgE4_lowF.txt", Z_1e6Pa_soc90_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.8

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.8
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc80_omgE4 = []
Z_1e6Pa_soc80_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_0Pa_soc80_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF", Z_1e6Pa_soc80_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF.txt", Z_1e6Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.7

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.7
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc70_omgE4 = []
Z_1e6Pa_soc70_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc70_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc70_omgE4_lowF", Z_1e6Pa_soc70_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc70_omgE4_lowF.txt", Z_1e6Pa_soc70_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.6

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.6
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc60_omgE4 = []
Z_1e6Pa_soc60_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc60_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc60_omgE4_lowF", Z_1e6Pa_soc60_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc60_omgE4_lowF.txt", Z_1e6Pa_soc60_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.5

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.5
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc50_omgE4 = []
Z_1e6Pa_soc50_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc50_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF", Z_1e6Pa_soc50_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF.txt", Z_1e6Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.4

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.4
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc40_omgE4 = []
Z_1e6Pa_soc40_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc40_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc40_omgE4_lowF", Z_1e6Pa_soc40_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc40_omgE4_lowF.txt", Z_1e6Pa_soc40_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.3

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.3
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc30_omgE4 = []
Z_1e6Pa_soc30_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc30_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF", Z_1e6Pa_soc30_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF.txt", Z_1e6Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.2

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.2
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc20_omgE4 = []
Z_1e6Pa_soc20_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc20_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc20_omgE4_lowF", Z_1e6Pa_soc20_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc20_omgE4_lowF.txt", Z_1e6Pa_soc20_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### SOC = 0.1

In [ ]:
hydrostatic_stress = 1e6 # Pa
initialSOC = 0.1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc10_omgE4 = []
Z_1e6Pa_soc10_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save results
# # sol_DFN.save("NMC_LiC6_sol_1e6Pa_soc10_omgE4_lowF.pkl")

# save to txt
np.save("NMC_LiC6_Z_1e6Pa_soc10_omgE4_lowF", Z_1e6Pa_soc10_omgE4)
np.savetxt("NMC_LiC6_Z_1e6Pa_soc10_omgE4_lowF.txt", Z_1e6Pa_soc10_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

### Vsi rezultati

In [ ]:
plt.plot(Z_1e6Pa_soc99_omgE4[:, 1], -Z_1e6Pa_soc99_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.99")
plt.plot(Z_1e6Pa_soc90_omgE4[:, 1], -Z_1e6Pa_soc90_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.9")
plt.plot(Z_1e6Pa_soc80_omgE4[:, 1], -Z_1e6Pa_soc80_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.8")
plt.plot(Z_1e6Pa_soc70_omgE4[:, 1], -Z_1e6Pa_soc70_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.7")
plt.plot(Z_1e6Pa_soc60_omgE4[:, 1], -Z_1e6Pa_soc60_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.6")
plt.plot(Z_1e6Pa_soc50_omgE4[:, 1], -Z_1e6Pa_soc50_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.5")
plt.plot(Z_1e6Pa_soc40_omgE4[:, 1], -Z_1e6Pa_soc40_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.4")
plt.plot(Z_1e6Pa_soc30_omgE4[:, 1], -Z_1e6Pa_soc30_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.3")
plt.plot(Z_1e6Pa_soc20_omgE4[:, 1], -Z_1e6Pa_soc20_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.2")
plt.plot(Z_1e6Pa_soc10_omgE4[:, 1], -Z_1e6Pa_soc10_omgE4[:, 2]+0.0, marker='o', linestyle='-', linewidth=1.5, markersize=4, label="SOC = 0.1")


# # Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('NMC+LiC6 ($\sigma_{\mathrm{h}}=1e6$ Pa)')
# plt.xlim(0.01, 0.025)
# plt.ylim(0.01, 0.025)
# plt.xlim(0, 0.1)
# plt.ylim(0, 0.1)

plt.legend(loc = "center right", bbox_to_anchor=(1.6, 0.5))

## Sprememba $i_0$ 

Spremenili bomo exchange current density na anodi in pogledali v katero smer se premakne DRT.
Izvedli bomo pri SOC 50%.

V datoteki ```Mohtat2020.py``` je spremenjena funkcija ```def graphite_electrolyte_exchange_current_density_PeymanMPM```. Za uvedbo spremembe je nato potrebno pognati datoteko ```install_pybamm.ipynb``` in restartati kernel.

In [ ]:
# import
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize
from joblib import Parallel, delayed
from scipy.signal import savgol_filter

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

# low_frequency = 0.0001 #lowF
low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc50_i010K = []
Z_0Pa_soc50_i010K = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress


# save to txt
np.save("NMC_LiC6 results/current_density/NMC_LiC6_Z_0Pa_soc50_i010K", Z_0Pa_soc50_i010K)
np.savetxt("NMC_LiC6 results/current_density/NMC_LiC6_Z_0Pa_soc50_i010K.txt", Z_0Pa_soc50_i010K)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

Primerjava pri različnih $i_0$.

In [ ]:
Z_0Pa_soc50 = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici.npy")
Z_0Pa_soc50_i02K = np.load("NMC_LiC6 results/current_density/NMC_LiC6_Z_0Pa_soc50_i02K.npy")
Z_0Pa_soc50_i010K = np.load("NMC_LiC6 results/current_density/NMC_LiC6_Z_0Pa_soc50_i010K.npy")
Z_0Pa_soc50_i0K2 = np.load("NMC_LiC6 results/current_density/NMC_LiC6_Z_0Pa_soc50_i0K2.npy")


plt.plot(Z_0Pa_soc50[:, 1], -Z_0Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$")

plt.plot(Z_0Pa_soc50_i02K[:, 1], -Z_0Pa_soc50_i02K[:, 2], marker='o', linestyle=':', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$x2K")    

plt.plot(Z_0Pa_soc50_i010K[:, 1], -Z_0Pa_soc50_i010K[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$x10K")

plt.plot(Z_0Pa_soc50_i0K2[:, 1], -Z_0Pa_soc50_i0K2[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$x0.5K")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                        "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4,
     },
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": 1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": 1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-4},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
# experiment = pybamm.Experiment(["Discharge at C/50 until 3.0 V"])
experiment_C = pybamm.Experiment(["Charge at C/50 until 4.2 V"])

solver = pybamm.IDAKLUSolver()  


# discharge battery
sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
# sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
#                               )
# sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
#                               )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_1 = sim_DFN_1.solve(initial_soc=1)
# end = time.time()
# print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_2 = sim_DFN_2.solve(initial_soc=1)
# end = time.time()
# print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_C = sim_DFN_0_C.solve(initial_soc=0)  
# end = time.time()
# print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_1_C = sim_DFN_1_C.solve(initial_soc=0)
# end = time.time()
# print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

# start = time.time()
# sol_DFN_2_C = sim_DFN_2_C.solve(initial_soc=0)
# end = time.time()
# print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD


# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
        #  SOC_1,
        # SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
        #  sol_DFN_1["Battery voltage [V]"].entries,
        # sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1e6 Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = 1e7 Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# # draw line at soc = 0.5
# plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# # draw line at 3.7156 V
# plt.axhline(y=3.7156, color='gray', linestyle='--', label='V = 3.7156 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
# plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))
plt.legend()
# plt.grid()



In [ ]:
# i_0 = sol_DFN["Positive electrode exchange current density [A.m-2]"].entries
# t = sol_DFN["Time [s]"].entries

# plt.plot (t, i_0, label=fr"$i_0$ = $i_0$x1K")
# plt.xlabel("Time [s]")
# plt.ylabel(r"$i_0$ [A.m$^{-2}$]")
# plt.title("Anode at C/50")
# plt.legend(loc='upper right', bbox_to_anchor=(1.4, 0.7))

# np.save("NMC_LiC6 results/current_density/i010K", i_0)
# np.save("NMC_LiC6 results/current_density/i010K_time", t)

In [ ]:
i_01A = np.load("NMC_LiC6 results/current_density/i01A.npy")
i_02A = np.load("NMC_LiC6 results/current_density/i02A.npy")
i_010A = np.load("NMC_LiC6 results/current_density/i010A.npy")
t_1A = np.load("NMC_LiC6 results/current_density/i01A_time.npy")
t_2A = np.load("NMC_LiC6 results/current_density/i02A_time.npy")
t_010A = np.load("NMC_LiC6 results/current_density/i010A_time.npy")



plt.figure(figsize=fig_size)
plt.plot(t_1A, i_01A[0, :], label="$i_0$ = $i_0$x1A")
plt.plot(t_2A, i_02A[0, :], label="$i_0$ = $i_0$x2A")
plt.plot(t_010A, i_010A[0, :], label="$i_0$ = $i_0$x10A")

plt.xlabel("Time [s]")
plt.ylabel(r"$i_0$ [A.m$^{-2}$]")
plt.title("Anode at C/50")
plt.legend(loc='upper right', bbox_to_anchor=(1.4, 0.7))


In [ ]:
i_0 = sol_DFN["Negative electrode exchange current density [A.m-2]"].entries
t = sol_DFN["Time [s]"].entries


plt.figure(figsize=fig_size)
plt.plot(t, i_0[0, :], label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa", linestyle='-')
plt.xlabel("Time [s]")
plt.ylabel(r"$i_0$ [A.m$^{-2}$]")
plt.title("Anode at C/50")

np.save("NMC_LiC6 results/current_density/i010K_anode", i_0)

In [ ]:
i_01K = np.load("NMC_LiC6 results/current_density/i01K.npy")
i_02K = np.load("NMC_LiC6 results/current_density/i02K.npy")
i_010K = np.load("NMC_LiC6 results/current_density/i010K.npy")
i_0K2 = np.load("NMC_LiC6 results/current_density/i0K2.npy")
t_1K = np.load("NMC_LiC6 results/current_density/i01K_time.npy")
t_2K = np.load("NMC_LiC6 results/current_density/i02K_time.npy")
t_010K = np.load("NMC_LiC6 results/current_density/i010K_time.npy")
t_K2 = np.load("NMC_LiC6 results/current_density/i0K2_time.npy")



plt.figure(figsize=fig_size)
plt.plot(t_1K, i_01K[0, :], label="$i_0$ = $i_0$x1K")
plt.plot(t_2K, i_02K[0, :], label="$i_0$ = $i_0$x2K")
plt.plot(t_010K, i_010K[0, :], label="$i_0$ = $i_0$x10K")
plt.plot(t_K2, i_0K2[0, :], label="$i_0$ = $i_0$x0.5K")


plt.xlabel("Time [s]")
plt.ylabel(r"$i_0$ [A.m$^{-2}$]")
plt.title("Anode at C/50")
plt.legend(loc='upper right', bbox_to_anchor=(1.45, 0.7))


Sprememba $i_0$ na katodi.

V datoteki ```Mohtat2020.py``` je spremenjena funkcija ```def NMC_electrolyte_exchange_current_density_PeymanMPM```. Za uvedbo spremembe je nato potrebno pognati datoteko ```install_pybamm.ipynb``` in restartati kernel.

In [ ]:
# import
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-6

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.1
high_frequency = 1000
Number_of_Nyquist_points = 100

#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
for i in range(0,len(freq_points)):

    freq = freq_points[i]
    
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = pybamm.ParameterValues("Mohtat2020_Mech") 
    param.update(
        {"Hydrostatic stress [Pa]": hydrostatic_stress,
         "Negative electrode partial molar volume [m3.mol-1]": omega},
        check_already_exists=False  
    )
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Z.append([freq, (est_amp/I_0)*np.cos(est_phase),(est_amp/I_0)*np.sin(est_phase)])

Z = np.array(Z)


In [ ]:
# rename variables
Z_0Pa_soc50_i01e3K = []
Z_0Pa_soc50_i01e3K = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress

# save results
# sol_DFN.save("NMC_LiC6_sol_0Pa_soc50_i01e3K.pkl")

# save to txt
np.save("NMC_LiC6_Z_0Pa_soc50_i01e3K", Z_0Pa_soc50_i01e3K)
np.savetxt("NMC_LiC6_Z_0Pa_soc50_i01e3K.txt", Z_0Pa_soc50_i01e3K)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

Primerjava pri različnih $i_0$.

In [ ]:
Z_0Pa_soc50 = np.load("NMC_LiC6_Z_0Pa_soc50.npy")
Z_0Pa_soc50_i02K = np.load("NMC_LiC6_Z_0Pa_soc50_i02K.npy")
Z_0Pa_soc50_i010K = np.load("NMC_LiC6_Z_0Pa_soc50_i010K.npy")
Z_0Pa_soc50_i01e3K = np.load("NMC_LiC6_Z_0Pa_soc50_i01e3K.npy")


plt.plot(Z_0Pa_soc50[:, 1], -Z_0Pa_soc50[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$")

plt.plot(Z_0Pa_soc50_i02K[:, 1], -Z_0Pa_soc50_i02K[:, 2], marker='o', linestyle=':', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$x2")    

plt.plot(Z_0Pa_soc50_i010K[:, 1], -Z_0Pa_soc50_i010K[:, 2], marker='o', linestyle='-.', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$x10")

plt.plot(Z_0Pa_soc50_i01e3K[:, 1], -Z_0Pa_soc50_i01e3K[:, 2], marker='o', linestyle='--', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = {hydrostatic_stress0:.0e} Pa, SOC = {initialSOC0:.2f}, $i_0$ = $i_0$x1e3")


# Labels and title
plt.xlabel('Re[Z] (Ω)')
plt.ylabel('-Im[Z] (Ω)')
plt.title('Nyquist Plot')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.show()

# Končni grafi

In [ ]:
# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 24,           # Osnovna velikost pisave
    'axes.titlesize': 24,      # Velikost naslova grafa
    'axes.labelsize': 24,      # Velikost pisave na osi
    'xtick.labelsize': 21,     # Velikost številk na x osi
    'ytick.labelsize': 21,     # Velikost številk na y osi
    'legend.fontsize': 22,     # Velikost pisave v legendi
})

## Brez Clericija

In [ ]:
Z_0Pa_soc30_omgE4 = np.load("NMC_LiC6_Z_0Pa_soc30_omgE4_lowF.npy") 
Z_1e6Pa_soc30_omgE4 = np.load("NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF.npy")
Z_1e7Pa_soc30_omgE4 = np.load("NMC_LiC6_Z_1e7Pa_soc30_omgE4_lowF.npy")

plt.plot(Z_0Pa_soc30_omgE4[:, 1], -Z_0Pa_soc30_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc30_omgE4[:, 1], -Z_1e6Pa_soc30_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc30_omgE4[:, 1], -Z_1e7Pa_soc30_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")


# Labels and title
# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.xlim(0, 0.025)
plt.ylim(0, 0.025)
plt.title('V = 3.64 V')
# plt.grid()

# save figure as eps
plt.savefig('NMC_LiC6_Nyquist_soc30_omgE4_lowF.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc50_omgE4 = np.load("NMC_LiC6_Z_0Pa_soc50_omgE4_lowF.npy") 
Z_1e6Pa_soc50_omgE4 = np.load("NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF.npy")
Z_1e7Pa_soc50_omgE4 = np.load("NMC_LiC6_Z_1e7Pa_soc50_omgE4_lowF.npy")


plt.plot(Z_0Pa_soc50_omgE4[:, 1], -Z_0Pa_soc50_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc50_omgE4[:, 1], -Z_1e6Pa_soc50_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc50_omgE4[:, 1], -Z_1e7Pa_soc50_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")


# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.72 V')
plt.xlim(0, 0.025)
plt.ylim(0, 0.025)
# plt.grid()

# save figure as eps
plt.savefig('NMC_LiC6_Nyquist_soc50_omgE4_lowF.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc80_omgE4 = np.load("NMC_LiC6_Z_0Pa_soc80_omgE4_lowF.npy")
Z_1e6Pa_soc80_omgE4 = np.load("NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF.npy")
Z_1e7Pa_soc80_omgE4 = np.load("NMC_LiC6_Z_1e7Pa_soc80_omgE4_lowF.npy")


plt.plot(Z_0Pa_soc80_omgE4[:, 1], -Z_0Pa_soc80_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc80_omgE4[:, 1], -Z_1e6Pa_soc80_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc80_omgE4[:, 1], -Z_1e7Pa_soc80_omgE4[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V= 3.98 V')
plt.xlim(0, 0.025)
plt.ylim(0, 0.025)
# plt.grid()

# save figure as eps
plt.savefig('NMC_LiC6_Nyquist_soc80_omgE4_lowF.eps', format='eps', dpi = 300, bbox_inches='tight')


## S Clericijem

In [ ]:
Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_lowF_Clerici.npy") 
Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF_Clerici.npy")
Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_lowF_Clerici.npy")

plt.plot(Z_0Pa_soc30_omgE4_Clerici[:, 1], -Z_0Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc30_omgE4_Clerici[:, 1], -Z_1e6Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc30_omgE4_Clerici[:, 1], -Z_1e7Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")


# Labels and title
# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
plt.title('V = 3.64 V')
# plt.grid()

# save figure as eps
plt.savefig('NMC_LiC6_Nyquist_soc30_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_lowF_Clerici.npy") 
Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF_Clerici.npy")
Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_lowF_Clerici.npy")


plt.plot(Z_0Pa_soc50_omgE4_Clerici[:, 1], -Z_0Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc50_omgE4_Clerici[:, 1], -Z_1e6Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc50_omgE4_Clerici[:, 1], -Z_1e7Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")


# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.72 V')
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
# plt.grid()

# save figure as eps
plt.savefig('NMC_LiC6_Nyquist_soc50_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_lowF_Clerici.npy") 
Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF_Clerici.npy")
Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_lowF_Clerici.npy")

plt.plot(Z_0Pa_soc80_omgE4_Clerici[:, 1], -Z_0Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc80_omgE4_Clerici[:, 1], -Z_1e6Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 1 MPa")

plt.plot(Z_1e7Pa_soc80_omgE4_Clerici[:, 1], -Z_1e7Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V= 3.98 V')
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
# plt.grid()

# save figure as eps
plt.savefig('NMC_LiC6_Nyquist_soc80_omgE4_lowF_Clerici.eps', format='eps', dpi = 300, bbox_inches='tight')


# Compressive stress

In [ ]:
# import
%matplotlib qt
import pybamm
import pybammeis 
import numpy as np
import matplotlib.pyplot as plt
import time
from ipywidgets import interact, FloatSlider
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from scipy import interpolate
from scipy import optimize
from joblib import Parallel, delayed
from scipy.signal import savgol_filter

# Shrani slike
save_fig = False 

# Shrani animacijo
save_ani = False

# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 20,           # Osnovna velikost pisave
    'axes.titlesize': 20,      # Velikost naslova grafa
    'axes.labelsize': 20,      # Velikost pisave na osi
    'xtick.labelsize': 18,     # Velikost številk na x osi
    'ytick.labelsize': 18,     # Velikost številk na y osi
    'legend.fontsize': 18,     # Velikost pisave v legendi
})

## SOC 30% $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5,
     },
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": -1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": -1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment([pybamm.step.string(
    "Discharge at C/50 until 2.75 V", period="1 s"
)
])

solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.3
plt.axvline(x=0.3, color='gray', linestyle='--', label='SOC = 0.3')

# draw line at 3.6391 V
plt.axhline(y=3.6391, color='gray', linestyle='--', label='V = 3.6391 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 30%.

In [ ]:
# find soc value closest to 0.3
target_soc = 0.3
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.3

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
    
    param = param0.copy() 
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc30_omgE4 = []
Z_0Pa_soc30_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_newModel", Z_0Pa_soc30_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_newModel.txt", Z_0Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = -1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param1.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc30_omgE4 = []
Z_1e6Pa_soc30_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_negStress_newModel", Z_1e6Pa_soc30_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_negStress_newModel.txt", Z_1e6Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = -1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param2.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc30_omgE4 = []
Z_1e7Pa_soc30_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_negStress_newModel", Z_1e7Pa_soc30_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_negStress_newModel.txt", Z_1e7Pa_soc30_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

In [ ]:
# full EIS
# Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_fullRange.npy") 
# Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_negStress_fullRange.npy")
# Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_negStress_fullRange.npy")

# semicircle EIS
Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_newModel.npy") 
Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_negStress_newModel.npy")
Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_negStress_newModel.npy")


plt.plot(Z_0Pa_soc30_omgE4_Clerici[:, 1], -Z_0Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc30_omgE4_Clerici[:, 1], -Z_1e6Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa")

plt.plot(Z_1e7Pa_soc30_omgE4_Clerici[:, 1], -Z_1e7Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.64 V')
plt.xlim(0, 0.015)
plt.ylim(0, 0.015)
# plt.grid()

# # save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc30_omgE4_Clerici_negStress_fullRange.eps', format='eps', dpi = 300, bbox_inches='tight')


## SOC 50% $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5,
     },
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": -1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": -1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5},
    check_already_exists=False  
)


var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment([pybamm.step.string(
    "Discharge at C/50 until 2.75 V", period="1 s"
)
])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(t_eval=t_eval, initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.5
plt.axvline(x=0.5, color='gray', linestyle='--', label='SOC = 0.5')

# draw line at 3.72 V
plt.axhline(y=3.72, color='gray', linestyle='--', label='V = 3.72 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 50%.

In [ ]:
# find soc value closest to 0.5
target_soc = 0.5
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.5
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #full range
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param0.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc50_omgE4 = []
Z_0Pa_soc50_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_newModel", Z_0Pa_soc50_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_newModel.txt", Z_0Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = -1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param1.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc50_omgE4 = []
Z_1e6Pa_soc50_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_negStress_newModel", Z_1e6Pa_soc50_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_negStress_newModel.txt", Z_1e6Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = -1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param2.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc50_omgE4 = []
Z_1e7Pa_soc50_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save to txt
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_negStress_newModel", Z_1e7Pa_soc50_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_negStress_newModel.txt", Z_1e7Pa_soc50_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")

## Vsi rezultati

In [ ]:
# full range EIS
# Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_fullRange.npy") 
# Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_negStress_fullRange.npy")
# Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_negStress_fullRange.npy")

# semicircle EIS
Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_newModel.npy") 
Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_negStress_newModel.npy")
Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_negStress_newModel.npy")

plt.plot(Z_0Pa_soc50_omgE4_Clerici[:, 1], -Z_0Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc50_omgE4_Clerici[:, 1], -Z_1e6Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa")

plt.plot(Z_1e7Pa_soc50_omgE4_Clerici[:, 1], -Z_1e7Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.72 V')
plt.xlim(0, 0.012)
plt.ylim(0, 0.012)
# plt.grid()

# # save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc50_omgE4_Clerici_negStress_fullRange.eps', format='eps', dpi = 300, bbox_inches='tight')


## SOC 80% $\Omega$ = 3.1e-4 $m^3/mol$

In [ ]:
model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )
param0 = pybamm.ParameterValues("Mohtat2020_Mech") 
param0.update(
    {"Hydrostatic stress [Pa]": 0,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5,
     },
    check_already_exists=False ,
    
)


param1 = pybamm.ParameterValues("Mohtat2020_Mech") 
param1.update(
    {"Hydrostatic stress [Pa]": -1e6,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5},
    check_already_exists=False  
)


param2 = pybamm.ParameterValues("Mohtat2020_Mech") 
param2.update(
    {"Hydrostatic stress [Pa]": -1e7,
     "Negative electrode partial molar volume [m3.mol-1]": 3.1e-6,
     "Positive electrode partial molar volume [m3.mol-1]": 1.25e-5},
    check_already_exists=False  
)

var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

experiment = pybamm.Experiment([pybamm.step.string(
    "Discharge at C/50 until 2.75 V", period="1 s"
)
])
solver = pybamm.IDAKLUSolver()  



sim_DFN_0 = pybamm.Simulation(model= model, parameter_values=param0, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_1 = pybamm.Simulation(model= model, parameter_values=param1, var_pts=var_pts, solver=solver, experiment=experiment 
                              )
sim_DFN_2 = pybamm.Simulation(model= model, parameter_values=param2, var_pts=var_pts, solver=solver,experiment=experiment 
                              )

######################
t_eval = [0, 3600]
# t_eval = np.linspace(0, 0.3, 1000)  
######################

start = time.time()
sol_DFN = sim_DFN_0.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_0 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_1 = sim_DFN_1.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_1 zaključena v {end - start:.2f} s.")

start = time.time()
sol_DFN_2 = sim_DFN_2.solve(initial_soc=1)
end = time.time()
print(f"Simulacija DFN_2 zaključena v {end - start:.2f} s.")

In [ ]:
Q_discharged = sol_DFN["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD = (Q_discharged / Q_max)
SOC = 1 - DOD

Q_discharged = sol_DFN_1["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_1 = (Q_discharged / Q_max)
SOC_1 = 1 - DOD_1

Q_discharged = sol_DFN_2["Discharge capacity [A.h]"].entries
Q_max = Q_discharged[-1] 
DOD_2 = (Q_discharged / Q_max)
SOC_2 = 1 - DOD_2

# x_val = [DOD, 
#          DOD_1,
#         DOD_2
#          ]

x_val = [SOC,
         SOC_1,
        SOC_2
         ]

y_val = [sol_DFN["Battery voltage [V]"].entries, 
         sol_DFN_1["Battery voltage [V]"].entries,
        sol_DFN_2["Battery voltage [V]"].entries,
         ]

# y_val = [sol_DFN["Battery open-circuit voltage [V]"].entries, 
#          sol_DFN_1["Battery open-circuit voltage [V]"].entries,
#         sol_DFN_2["Battery open-circuit voltage [V]"].entries
#          ]


plt.figure(figsize=fig_size)

for i in range(len(x_val)):
    plt.plot(x_val[i], y_val[i], label=[fr"$\sigma_{{\mathrm{{h}}}}$ = {param0['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param1['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        fr"$\sigma_{{\mathrm{{h}}}}$ = {param2['Hydrostatic stress [Pa]']:.0e} Pa", 
                                        ][i],
                                linestyle=['-', '--', '-.' ][i]

                                        )
    
# draw line at soc = 0.8
plt.axvline(x=0.8, color='gray', linestyle='--', label='SOC = 0.8')

# draw line at 3.9803 V
plt.axhline(y=3.9803, color='gray', linestyle='--', label='V = 3.9803 V')

plt.xlabel("SOC [/]")
plt.ylabel("V [V]")
plt.legend(loc="upper right", bbox_to_anchor=(1.5, 0.9))

Pogledamo pri kateri napetosti neobremenjene celice je SOC = 80%.

In [ ]:
# find soc value closest to 0.8
target_soc = 0.8
index = (np.abs(SOC - target_soc)).argmin() 
print(f"Closest SOC to {target_soc} is {SOC[index]:.4f} at index {index}.")
print(f"Corresponding Voltage: {sol_DFN['Voltage [V]'].entries[index]:.4f} V.")

Pogledamo pri katerem SOC so obremenjene celice za enako napetost.

In [ ]:
# target voltage
target_voltage_1 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_1 = (np.abs(sol_DFN_1["Voltage [V]"].entries - target_voltage_1 )).argmin()
target_soc_1 = SOC_1[index_voltage_1]

print(f"For hydrostatic stress of {param1['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param1['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_1['Voltage [V]'].entries[index_voltage_1]:.4f} V at index {index_voltage_1}.")
print(f"Corresponding SOC: {target_soc_1:.4f}.")

In [ ]:
# target voltage
target_voltage_2 = sol_DFN['Voltage [V]'].entries[index]
index_voltage_2 = (np.abs(sol_DFN_2["Voltage [V]"].entries - target_voltage_2 )).argmin()
target_soc_2 = SOC_2[index_voltage_2]

print(f"For hydrostatic stress of {param2['Hydrostatic stress [Pa]']:.0e} Pa and partial molar volume of {param2['Negative electrode partial molar volume [m3.mol-1]']:.1e} m3.mol-1:")
print(f"Corrected voltage: {sol_DFN_2['Voltage [V]'].entries[index_voltage_2]:.4f} V at index {index_voltage_2}.")
print(f"Corresponding SOC: {target_soc_2:.4f}.")

#### $\sigma_h$ = 0 Pa

Izračunamo EIS

In [ ]:
hydrostatic_stress = 0 # Pa
initialSOC = 0.8
omega = 3.1e-4

######################################
I_0 = 0.1 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 20 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #full range
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param0.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_0Pa_soc80_omgE4 = []
Z_0Pa_soc80_omgE4 = Z
initialSOC0 = initialSOC
hydrostatic_stress0 = hydrostatic_stress
I0_0 = I_0
w_0 = Number_of_waves

# save to txt
# np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_fullRange", Z_0Pa_soc80_omgE4)
# np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_fullRange.txt", Z_0Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")
np.save("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_newModel", Z_0Pa_soc80_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_newModel.txt", Z_0Pa_soc80_omgE4)

#### $\sigma_h$ = 1e6 Pa

In [ ]:
hydrostatic_stress = -1e6 # Pa
initialSOC = target_soc_1
omega = 3.1e-4

######################################
I_0 = 0.4 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param1.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e6Pa_soc80_omgE4 = []
Z_1e6Pa_soc80_omgE4 = Z
initialSOC1 = initialSOC
hydrostatic_stress1 = hydrostatic_stress
I0_1 = I_0
w_1 = Number_of_waves

# save to txt
# np.save("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_fullRange", Z_1e6Pa_soc80_omgE4)
# np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_fullRange.txt", Z_1e6Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_newModel", Z_1e6Pa_soc80_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_newModel.txt", Z_1e6Pa_soc80_omgE4)

#### $\sigma_h$ = 1e7 Pa

In [ ]:
hydrostatic_stress = -1e7 # Pa
initialSOC = target_soc_2
omega = 3.1e-4

######################################
I_0 = 0.9 #sampling current amplitude
N = 1000 #precission (higher value results in more precise impedance calculation at the expense of longer computational time)
Number_of_waves = 100 #number of sine waves used to determine the impedance (higher value results in more precise impedance calculation at the expense of longer computational time)

low_frequency = 0.0001 #fullRange
# low_frequency = 0.1 #semicircle
high_frequency = 1000
Number_of_Nyquist_points = 100


In [ ]:
#initialisation of table where impedances will be stored
Z = []

#freq_points = [0.0001, 0.001, 0.01, 0.1, 1] #for testing
freq_points = np.logspace(np.log10(low_frequency), np.log10(high_frequency), Number_of_Nyquist_points)

#for loop acros all measured frequencies
def compute_impedance_point(freq):
     
    t = np.linspace(0, Number_of_waves / (freq), N) #preparing table with simulation times (30 wavelengths of given frequency freq)
   
    #Generation of the current signal used for sampling
    sim_I_L = I_0*np.sin(freq*t) # creating sinusoidal wave data stored in a form of list with the precission N
    # TUKAJ JE TREBA APLICIRAN SIMULACIJSKI TOK (sinusni) SPRAVITI V PYTHON FUNKCIJO, KI JO PYBAMM RAZUME - PRED TEM JE TO TABELA sim_I_L
    # To je prekopirano iz https://docs.pybamm.org/en/stable/source/examples/notebooks/parameterization/change-input-current.html
    def my_fun(I0, freq):
        def current(t):
            return I0 * pybamm.sin(2 * np.pi * freq * t)
        return current
    

    #TUKAJ POŽENEŠ SVOJO PYBAMM SIMULACIJO (TRENUTNO JE TUKAJ OSNOVNA "PLACEHOLDER SIMULACIJA"):


    model = pybamm.lithium_ion.DFN(options={"surface form": "differential",
                                                "particle": "Fickian diffusion",
                                                },
                                        )

    param = param2.copy()
    param["Current function [A]"] = my_fun(I_0, freq)

    var_pts = {
    "x_n": 30,  # negative electrode
    "x_s": 20,  # separator
    "x_p": 30,  # positive electrode
    "r_n": 25,  # negative particle
    "r_p": 25,  # positive particle
}

    # experiment = pybamm.Experiment(["Discharge at 1 C until 3.0 V"])
    solver = pybamm.IDAKLUSolver()  

    sim_DFN = pybamm.Simulation(model= model, parameter_values=param, var_pts=var_pts, solver=solver, #experiment=experiment 
                                )

    sol_DFN = sim_DFN.solve(t_eval=t, initial_soc=initialSOC)

    simulated_U_L =  sol_DFN["Voltage [V]"].entries #VRNEMO IZRAČUNANO NAPETOST


    #SLEDI FITTING RUTINA, KI PREBERE AMPLITUDO IN FAZNI ZAMIK IZRAČUNANE NAPETOSTI
    #Preparing guess that will be used as a starting point for the fitting function
    guess_mean = np.mean(simulated_U_L) #guess for the mean voltage
    guess_phase = 0 #gues for the phase shift is set to 0
    guess_amp = np.std(simulated_U_L)/(2**0.5)/(2**0.5) #guess for the amplitude is calculated as standard deviation of simulated data


    # Defining the function to optimize, in this case, we want to minimize the difference between the simulated data and our "guessed" parameters
    optimize_func = lambda x: -x[0]*np.sin(2 * np.pi * freq * sol_DFN["Time [s]"].entries + x[1]) + x[2] - simulated_U_L
    #fitting - estimating amplitude, phase shift and mean of the simulated voltage function
    est_amp, est_phase, est_mean = optimize.leastsq(optimize_func, [guess_amp, guess_phase, guess_mean])[0]

   
    #IZRAČUNANE VREDNOSTI UPORABIMO ZA IMEPDANCO Z
    #Calculating impedance of the system for given frequency from est_amp and ext_phase and storing it in the table Z
    Zre = (est_amp / I_0) * np.cos(est_phase)
    Zim = (est_amp / I_0) * np.sin(est_phase)

    return [freq, Zre, Zim]

Z = Parallel(n_jobs=6)(
        delayed(compute_impedance_point)(f) 
        for f in freq_points
    )

Z = np.array(Z)

In [ ]:
# rename variables
Z_1e7Pa_soc80_omgE4 = []
Z_1e7Pa_soc80_omgE4 = Z
initialSOC2 = initialSOC
hydrostatic_stress2 = hydrostatic_stress
I0_2 = I_0
w_2 = Number_of_waves

# save to txt
# np.save("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_fullRange", Z_1e7Pa_soc80_omgE4)
# np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_fullRange.txt", Z_1e7Pa_soc80_omgE4)#, header="Frequency(Hz) Re(Z)(Ohm) Im(Z)(Ohm)")
np.save("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_newModel", Z_1e7Pa_soc80_omgE4)
np.savetxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_newModel.txt", Z_1e7Pa_soc80_omgE4)

## Vsi rezultati

In [ ]:
# # full EIS
# Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_fullRange.npy") 
# Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_fullRange.npy")
# Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_fullRange.npy")

# semicircle EIS
Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_newModel.npy") 
Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_newModel.npy")
Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_newModel.npy")



plt.plot(Z_0Pa_soc80_omgE4_Clerici[:, 1], -Z_0Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc80_omgE4_Clerici[:, 1], -Z_1e6Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa")

plt.plot(Z_1e7Pa_soc80_omgE4_Clerici[:, 1], -Z_1e7Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.98 V')
plt.xlim(0, 0.012)
plt.ylim(0, 0.012)
# plt.grid()

# # save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc80_omgE4_Clerici_negStress_fullRange.eps', format='eps', dpi = 300, bbox_inches='tight')


# Končni grafi

In [ ]:
# Formatiranje grafov
# Nastavitev velikosti grafa (širina, višina v inch-ih)
fig_size = figsize=(8, 6)  

# Nastavitev velikosti pisave
plt.rcParams.update({
    'font.size': 24,           # Osnovna velikost pisave
    'axes.titlesize': 24,      # Velikost naslova grafa
    'axes.labelsize': 24,      # Velikost pisave na osi
    'xtick.labelsize': 21,     # Velikost številk na x osi
    'ytick.labelsize': 21,     # Velikost številk na y osi
    'legend.fontsize': 22,     # Velikost pisave v legendi
})

In [ ]:
Z_0Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_newModel.npy") 
Z_1e6Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_negStress_newModel.npy")
Z_1e7Pa_soc30_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_negStress_newModel.npy")

plt.plot(Z_0Pa_soc30_omgE4_Clerici[:, 1], -Z_0Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc30_omgE4_Clerici[:, 1], -Z_1e6Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa")

plt.plot(Z_1e7Pa_soc30_omgE4_Clerici[:, 1], -Z_1e7Pa_soc30_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa")


# Labels and title
# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
plt.title('V = 3.64 V')
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc30_omgE4_Clerici_negStress.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_newModel.npy") 
Z_1e6Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_negStress_newModel.npy")
Z_1e7Pa_soc50_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_negStress_newModel.npy")


plt.plot(Z_0Pa_soc50_omgE4_Clerici[:, 1], -Z_0Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc50_omgE4_Clerici[:, 1], -Z_1e6Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa")

plt.plot(Z_1e7Pa_soc50_omgE4_Clerici[:, 1], -Z_1e7Pa_soc50_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa")


# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')

# plt.legend(loc='center right', bbox_to_anchor=(1.6, 0.5))
# plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V = 3.72 V')
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc50_omgE4_Clerici_negStress.eps', format='eps', dpi = 300, bbox_inches='tight')


In [ ]:
Z_0Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_newModel.npy") 
Z_1e6Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_negStress_newModel.npy")
Z_1e7Pa_soc80_omgE4_Clerici = np.load("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_negStress_newModel.npy")

plt.plot(Z_0Pa_soc80_omgE4_Clerici[:, 1], -Z_0Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = 0 Pa")

plt.plot(Z_1e6Pa_soc80_omgE4_Clerici[:, 1], -Z_1e6Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4, 
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -1 MPa")

plt.plot(Z_1e7Pa_soc80_omgE4_Clerici[:, 1], -Z_1e7Pa_soc80_omgE4_Clerici[:, 2], marker='o', linestyle='-', linewidth=1.5, markersize=4,
        label=fr"$\sigma_{{\mathrm{{h}}}}$ = -10 MPa")

# Labels and title
plt.xlabel('Re(Z) [Ω]')
plt.ylabel('-Im(Z) [Ω]')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))
plt.title('V= 3.98 V')
plt.xlim(0, 0.020)
plt.ylim(0, 0.020)
# plt.grid()

# save figure as eps
# plt.savefig('NMC_LiC6_Nyquist_soc80_omgE4_Clerici_negStress.eps', format='eps', dpi = 300, bbox_inches='tight')


# DRT

In [ ]:
import numpy as np
%matplotlib qt
import matplotlib.pyplot as plt

## Brez Clericija

In [ ]:
# read from txt
Z_0Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6_Z_0Pa_soc30_omgE4_lowF_DRT.txt")
Z_0Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6_Z_0Pa_soc50_omgE4_lowF_DRT.txt")
Z_0Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6_Z_0Pa_soc80_omgE4_lowF_DRT.txt")

Z_1e6Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6_Z_1e6Pa_soc30_omgE4_lowF_DRT.txt")
Z_1e6Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6_Z_1e6Pa_soc50_omgE4_lowF_DRT.txt")
Z_1e6Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6_Z_1e6Pa_soc80_omgE4_lowF_DRT.txt")

Z_1e7Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6_Z_1e7Pa_soc30_omgE4_lowF_DRT.txt")
Z_1e7Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6_Z_1e7Pa_soc50_omgE4_lowF_DRT.txt")
Z_1e7Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6_Z_1e7Pa_soc80_omgE4_lowF_DRT.txt")

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc30_omgE4[:, 0], Z_0Pa_soc30_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc30_omgE4[:, 0], Z_1e6Pa_soc30_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc30_omgE4[:, 0], Z_1e7Pa_soc30_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.64 V')
# plt.grid(True, which='both')
# plt.legend()

# save figure as eps
plt.savefig('NMC_LiC6_DRT_soc30_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc50_omgE4[:, 0], Z_0Pa_soc50_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc50_omgE4[:, 0], Z_1e6Pa_soc50_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc50_omgE4[:, 0], Z_1e7Pa_soc50_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.72 V')
# plt.grid(True, which='both')
# plt.legend()

# save as eps
plt.savefig('NMC_LiC6_DRT_soc50_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc80_omgE4[:, 0], Z_0Pa_soc80_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc80_omgE4[:, 0], Z_1e6Pa_soc80_omgE4[:, 1], label='1 MPa')
plt.plot(Z_1e7Pa_soc80_omgE4[:, 0], Z_1e7Pa_soc80_omgE4[:, 1], label='10 MPa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.98 V')
# plt.grid(True, which='both')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# save as eps
plt.savefig('NMC_LiC6_DRT_soc80_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

## S Clericijem

In [ ]:
# read from txt
Z_0Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_DRT.txt")
Z_0Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_DRT.txt")
Z_0Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_DRT.txt")

Z_1e6Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc30_omgE4_Clerici_DRT.txt")
Z_1e6Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc50_omgE4_Clerici_DRT.txt")
Z_1e6Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e6Pa_soc80_omgE4_Clerici_DRT.txt")

Z_1e7Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc30_omgE4_Clerici_DRT.txt")
Z_1e7Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc50_omgE4_Clerici_DRT.txt")
Z_1e7Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1e7Pa_soc80_omgE4_Clerici_DRT.txt")

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc30_omgE4[:, 0], Z_0Pa_soc30_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc30_omgE4[:, 0], Z_1e6Pa_soc30_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc30_omgE4[:, 0], Z_1e7Pa_soc30_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.64 V')
# plt.grid(True, which='both')
# plt.legend()

# save figure as eps
plt.savefig('NMC_LiC6_DRT_soc30_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc50_omgE4[:, 0], Z_0Pa_soc50_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc50_omgE4[:, 0], Z_1e6Pa_soc50_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc50_omgE4[:, 0], Z_1e7Pa_soc50_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.72 V')
# plt.grid(True, which='both')
# plt.legend()

# save as eps
plt.savefig('NMC_LiC6_DRT_soc50_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.plot(Z_0Pa_soc80_omgE4[:, 0], Z_0Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 0 Pa')
plt.plot(Z_1e6Pa_soc80_omgE4[:, 0], Z_1e6Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 1 MPa')
plt.plot(Z_1e7Pa_soc80_omgE4[:, 0], Z_1e7Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 10 MPa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.98 V')
# plt.grid(True, which='both')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# save as eps
plt.savefig('NMC_LiC6_DRT_soc80_omgE4.eps', format='eps', dpi=300, bbox_inches='tight')

## Compressive stress

In [ ]:
# read from txt
Z_0Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc30_omgE4_Clerici_negStress_DRT.txt")
Z_0Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc50_omgE4_Clerici_negStress_DRT.txt")
Z_0Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_0Pa_soc80_omgE4_Clerici_negStress_DRT.txt")

Z_1e6Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1MPa_soc30_omgE4_Clerici_negStress_DRT.txt")
Z_1e6Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1MPa_soc50_omgE4_Clerici_negStress_DRT.txt")
Z_1e6Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_1MPa_soc80_omgE4_Clerici_negStress_DRT.txt")

Z_1e7Pa_soc30_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_10MPa_soc30_omgE4_Clerici_negStress_DRT.txt")
Z_1e7Pa_soc50_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_10MPa_soc50_omgE4_Clerici_negStress_DRT.txt")
Z_1e7Pa_soc80_omgE4 = np.loadtxt("NMC_LiC6 results/NMC_LiC6_Z_10MPa_soc80_omgE4_Clerici_negStress_DRT.txt")

In [ ]:
# Plot DRT results
plt.figure(figsize=fig_size)

plt.plot(Z_0Pa_soc30_omgE4[:, 0], Z_0Pa_soc30_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc30_omgE4[:, 0], Z_1e6Pa_soc30_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc30_omgE4[:, 0], Z_1e7Pa_soc30_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.64 V')
# plt.grid(True, which='both')
# plt.legend()

# save figure as eps
plt.savefig('NMC_LiC6_DRT_soc30_omgE4_negStress.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.figure(figsize=fig_size)

plt.plot(Z_0Pa_soc50_omgE4[:, 0], Z_0Pa_soc50_omgE4[:, 1], label='0 Pa')
plt.plot(Z_1e6Pa_soc50_omgE4[:, 0], Z_1e6Pa_soc50_omgE4[:, 1], label='1e6 Pa')
plt.plot(Z_1e7Pa_soc50_omgE4[:, 0], Z_1e7Pa_soc50_omgE4[:, 1], label='1e7 Pa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.72 V')
# plt.grid(True, which='both')
# plt.legend()

# save as eps
plt.savefig('NMC_LiC6_DRT_soc50_omgE4_negStress.eps', format='eps', dpi=300, bbox_inches='tight')

In [ ]:
# Plot DRT results
plt.figure(figsize=fig_size)
plt.plot(Z_0Pa_soc80_omgE4[:, 0], Z_0Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 0 Pa')
plt.plot(Z_1e6Pa_soc80_omgE4[:, 0], Z_1e6Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 1 MPa')
plt.plot(Z_1e7Pa_soc80_omgE4[:, 0], Z_1e7Pa_soc80_omgE4[:, 1], label='$\sigma_{{\mathrm{{h}}}}$ = 10 MPa')

# add logarithmic scale to x axis
plt.xscale('log')

# add labels and titlež
plt.xlabel(r'$\tau$ [s]')
plt.ylabel('$\gamma$  [/]')
plt.title('V = 3.98 V')
# plt.grid(True, which='both')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2))

# save as eps
plt.savefig('NMC_LiC6_DRT_soc80_omgE4_negStress.eps', format='eps', dpi=300, bbox_inches='tight')